In [ ]:
!pip install requests beautifulsoup4 lxml loguru langchain-groq langchain_nvidia_ai_endpoints -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.4 MB/s eta 0:00:00


In [2]:
import os
SEC_API_KEY_OLD = os.environ.get("SEC_API_KEY", "f589b2bc120b3ddb2d2835e86c694d83070a3445a74ee2d795f64abc2fa1370e")
SEC_API_KEY = os.environ.get("SEC_API_KEY", "c67770ee973fe24ae572720db4a152071d0f9c8eb1c2713b0cdc225089ad4989")
SEC_API_KEY = os.environ.get("SEC_API_KEY", "0e88bf462569faad922fa778a460da4bbe6d3d156f02075b64191e4d136b0fd0")




import os
import re
import json
import time
import hashlib
import logging
from datetime import datetime, timedelta
from html.parser import HTMLParser
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv(override=True)

load_dotenv(override=True)

# ─── NVIDIA LLM ─────────────────────────────────────────────────────────────────
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from google.colab import userdata

# Get API key from Colab secrets
nvidia_api_key = userdata.get('NVIDIA_API_KEY')  # or 'NVAPI_KEY'

MODEL = "moonshotai/kimi-k2.6"  # or "openai/gpt-oss-120b", "meta/llama3-70b", etc.

def get_llm(temperature: float = 0.2) -> ChatNVIDIA:
    return ChatNVIDIA(
        model=MODEL,
        api_key=nvidia_api_key,
        temperature=temperature,
        top_p=1,
        max_completion_tokens=16384,
    )




In [3]:
load_dotenv(override=True)

# ─── NVIDIA LLM ─────────────────────────────────────────────────────────────────
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from google.colab import userdata

nvidia_api_key = userdata.get('CHATGPT')

MODEL = "openai/gpt-oss-20b"

def get_llm(temperature: float = 0.2) -> ChatNVIDIA:
    return ChatNVIDIA(
        model=MODEL,
        api_key=nvidia_api_key,
        temperature=temperature,
        top_p=1,
        max_tokens=4096,
    )

# Non-streaming version
client = get_llm()
response = client.invoke([{"content": "what is RAG? short answer, 5-6 sentences", "role": "user"}])
print(response.content)

/tmp/ipykernel_3715/4009438480.py:12: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  return ChatNVIDIA(


RAG, or Retrieval‑Augmented Generation, is a hybrid natural‑language‑processing framework that combines a language model with an external knowledge source. Instead of generating text solely from its internal parameters, a RAG system first retrieves relevant documents or passages from a large corpus (often using dense or sparse vector search). These retrieved snippets are then fed into the language model as additional context, allowing the model to produce more accurate, up‑to‑date, or fact‑based responses. The approach is especially useful for tasks like question answering, summarization, or dialogue where up‑to‑date information is crucial. By decoupling knowledge storage from the generative model, RAG can adapt to new data without retraining the entire system.


In [3]:
load_dotenv(override=True)

# ─── NVIDIA LLM ─────────────────────────────────────────────────────────────────
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from google.colab import userdata

nvidia_api_key = userdata.get('CHATGPT')

MODEL = "openai/gpt-oss-120b"

def get_llm(temperature: float = 0.2) -> ChatNVIDIA:
    return ChatNVIDIA(
        model=MODEL,
        api_key=nvidia_api_key,
        temperature=temperature,
        top_p=1,
        max_tokens=4096,
    )

# Non-streaming version
client = get_llm()
response = client.invoke([{"content": "what is RAG? short answer, 5-6 sentences", "role": "user"}])
print(response.content)

/tmp/ipykernel_29732/1389142286.py:12: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  return ChatNVIDIA(


Retrieval‑Augmented Generation (RAG) is a hybrid AI architecture that combines a **retriever**—a search component that pulls relevant documents or passages from an external knowledge base—with a **generator**—a large language model that synthesizes those retrieved texts into a coherent response.  

1. When a user query arrives, the retriever (often a dense vector‑based or BM25 search) quickly fetches the top‑k most pertinent chunks of information.  
2. The generator then conditions its language model on both the original query and the retrieved chunks, allowing it to ground its output in factual data rather than relying solely on memorized knowledge.  
3. This setup improves factual accuracy, reduces hallucinations, and lets the system stay up‑to‑date simply by refreshing the underlying document store.  

RAG is widely used for open‑domain question answering, chat assistants, and enterprise search where real‑time, verifiable information is essential.


In [4]:
def get_nasdaq100_tickers() -> list[dict]:
    """Return accurate list of exactly 500 S&P 500 companies."""
    return [
        # Technology (78)
        {"ticker": "AAPL", "name": "Apple Inc"},
        {"ticker": "MSFT", "name": "Microsoft Corp"},
        {"ticker": "NVDA", "name": "NVIDIA Corp"},
        {"ticker": "AVGO", "name": "Broadcom Inc"},
        {"ticker": "ORCL", "name": "Oracle Corp"},
        {"ticker": "ADBE", "name": "Adobe Inc"},
        {"ticker": "CRM", "name": "Salesforce Inc"},
        {"ticker": "AMD", "name": "Advanced Micro Devices Inc"},
        {"ticker": "INTC", "name": "Intel Corp"},
        {"ticker": "QCOM", "name": "Qualcomm Inc"},
        {"ticker": "TXN", "name": "Texas Instruments Inc"},
        {"ticker": "IBM", "name": "International Business Machines Corp"},
        {"ticker": "CSCO", "name": "Cisco Systems Inc"},
        {"ticker": "NOW", "name": "ServiceNow Inc"},
        {"ticker": "PANW", "name": "Palo Alto Networks Inc"},
        {"ticker": "INTU", "name": "Intuit Inc"},
        {"ticker": "SNOW", "name": "Snowflake Inc"},
        {"ticker": "CRWD", "name": "CrowdStrike Holdings Inc"},
        {"ticker": "DELL", "name": "Dell Technologies Inc"},
        {"ticker": "HPQ", "name": "HP Inc"},
        {"ticker": "ADI", "name": "Analog Devices Inc"},
        {"ticker": "MU", "name": "Micron Technology Inc"},
        {"ticker": "LRCX", "name": "Lam Research Corp"},
        {"ticker": "KLAC", "name": "KLA Corp"},
        {"ticker": "AMAT", "name": "Applied Materials Inc"},
        {"ticker": "MCHP", "name": "Microchip Technology Inc"},
        {"ticker": "ON", "name": "ON Semiconductor Corp"},
        {"ticker": "FTNT", "name": "Fortinet Inc"},
        {"ticker": "NET", "name": "Cloudflare Inc"},
        {"ticker": "ZS", "name": "Zscaler Inc"},
        {"ticker": "DDOG", "name": "Datadog Inc"},
        {"ticker": "MDB", "name": "MongoDB Inc"},
        {"ticker": "TEAM", "name": "Atlassian Corp"},
        {"ticker": "WDAY", "name": "Workday Inc"},
        {"ticker": "ADSK", "name": "Autodesk Inc"},
        {"ticker": "CDNS", "name": "Cadence Design Systems Inc"},
        {"ticker": "SNPS", "name": "Synopsys Inc"},
        {"ticker": "ROP", "name": "Roper Technologies Inc"},
        {"ticker": "TTD", "name": "Trade Desk Inc"},
        {"ticker": "HUBS", "name": "HubSpot Inc"},
        {"ticker": "AKAM", "name": "Akamai Technologies Inc"},
        {"ticker": "VRSN", "name": "Verisign Inc"},
        {"ticker": "FFIV", "name": "F5 Inc"},
        {"ticker": "JNPR", "name": "Juniper Networks Inc"},
        {"ticker": "MSI", "name": "Motorola Solutions Inc"},
        {"ticker": "HPE", "name": "Hewlett Packard Enterprise Co"},
        {"ticker": "CDW", "name": "CDW Corp"},
        {"ticker": "GLW", "name": "Corning Inc"},
        {"ticker": "APH", "name": "Amphenol Corp"},
        {"ticker": "TEL", "name": "TE Connectivity Ltd"},
        {"ticker": "JBL", "name": "Jabil Inc"},
        {"ticker": "FLEX", "name": "Flex Ltd"},
        {"ticker": "STX", "name": "Seagate Technology Holdings PLC"},
        {"ticker": "WDC", "name": "Western Digital Corp"},
        {"ticker": "NTAP", "name": "NetApp Inc"},
        {"ticker": "KEYS", "name": "Keysight Technologies Inc"},
        {"ticker": "MPWR", "name": "Monolithic Power Systems Inc"},
        {"ticker": "SWKS", "name": "Skyworks Solutions Inc"},
        {"ticker": "NXPI", "name": "NXP Semiconductors NV"},
        {"ticker": "ANET", "name": "Arista Networks Inc"},
        {"ticker": "IT", "name": "Gartner Inc"},
        {"ticker": "GEN", "name": "Gen Digital Inc"},
        {"ticker": "EPAM", "name": "EPAM Systems Inc"},
        {"ticker": "TYL", "name": "Tyler Technologies Inc"},
        {"ticker": "PTC", "name": "PTC Inc"},
        {"ticker": "ANSS", "name": "ANSYS Inc"},
        {"ticker": "SSNC", "name": "SS&C Technologies Holdings Inc"},
        {"ticker": "ZM", "name": "Zoom Video Communications Inc"},
        {"ticker": "DOCU", "name": "DocuSign Inc"},
        {"ticker": "OKTA", "name": "Okta Inc"},
        {"ticker": "SMCI", "name": "Super Micro Computer Inc"},
        {"ticker": "PLTR", "name": "Palantir Technologies Inc"},
        {"ticker": "FSLR", "name": "First Solar Inc"},
        {"ticker": "ENPH", "name": "Enphase Energy Inc"},
        {"ticker": "TER", "name": "Teradyne Inc"},
        {"ticker": "VSH", "name": "Vishay Intertechnology Inc"},
        {"ticker": "GL", "name": "Globe Life Inc"},
        {"ticker": "ZBRA", "name": "Zebra Technologies Corp"},

        # Financials (70)
        {"ticker": "BRK.B", "name": "Berkshire Hathaway Inc"},
        {"ticker": "JPM", "name": "JPMorgan Chase & Co"},
        {"ticker": "V", "name": "Visa Inc"},
        {"ticker": "MA", "name": "Mastercard Inc"},
        {"ticker": "BAC", "name": "Bank of America Corp"},
        {"ticker": "WFC", "name": "Wells Fargo & Co"},
        {"ticker": "GS", "name": "Goldman Sachs Group Inc"},
        {"ticker": "MS", "name": "Morgan Stanley"},
        {"ticker": "C", "name": "Citigroup Inc"},
        {"ticker": "AXP", "name": "American Express Co"},
        {"ticker": "BLK", "name": "BlackRock Inc"},
        {"ticker": "SCHW", "name": "Charles Schwab Corp"},
        {"ticker": "SPGI", "name": "S&P Global Inc"},
        {"ticker": "ICE", "name": "Intercontinental Exchange Inc"},
        {"ticker": "MCO", "name": "Moody's Corp"},
        {"ticker": "CME", "name": "CME Group Inc"},
        {"ticker": "PNC", "name": "PNC Financial Services Group Inc"},
        {"ticker": "USB", "name": "US Bancorp"},
        {"ticker": "TFC", "name": "Truist Financial Corp"},
        {"ticker": "COF", "name": "Capital One Financial Corp"},
        {"ticker": "BK", "name": "Bank of New York Mellon Corp"},
        {"ticker": "STT", "name": "State Street Corp"},
        {"ticker": "FITB", "name": "Fifth Third Bancorp"},
        {"ticker": "KEY", "name": "KeyCorp"},
        {"ticker": "CFG", "name": "Citizens Financial Group Inc"},
        {"ticker": "HBAN", "name": "Huntington Bancshares Inc"},
        {"ticker": "RF", "name": "Regions Financial Corp"},
        {"ticker": "SYF", "name": "Synchrony Financial"},
        {"ticker": "DFS", "name": "Discover Financial Services"},
        {"ticker": "MTB", "name": "M&T Bank Corp"},
        {"ticker": "FIS", "name": "Fidelity National Information Services Inc"},
        {"ticker": "FISV", "name": "Fiserv Inc"},
        {"ticker": "GPN", "name": "Global Payments Inc"},
        {"ticker": "PGR", "name": "Progressive Corp"},
        {"ticker": "ALL", "name": "Allstate Corp"},
        {"ticker": "TRV", "name": "Travelers Companies Inc"},
        {"ticker": "AIG", "name": "American International Group Inc"},
        {"ticker": "MET", "name": "MetLife Inc"},
        {"ticker": "PRU", "name": "Prudential Financial Inc"},
        {"ticker": "CB", "name": "Chubb Ltd"},
        {"ticker": "AFL", "name": "Aflac Inc"},
        {"ticker": "AJG", "name": "Arthur J Gallagher & Co"},
        {"ticker": "MMC", "name": "Marsh & McLennan Companies Inc"},
        {"ticker": "WTW", "name": "Willis Towers Watson PLC"},
        {"ticker": "BRO", "name": "Brown & Brown Inc"},
        {"ticker": "CINF", "name": "Cincinnati Financial Corp"},
        {"ticker": "WRB", "name": "WR Berkley Corp"},
        {"ticker": "PFG", "name": "Principal Financial Group Inc"},
        {"ticker": "TROW", "name": "T Rowe Price Group Inc"},
        {"ticker": "BEN", "name": "Franklin Resources Inc"},
        {"ticker": "AMP", "name": "Ameriprise Financial Inc"},
        {"ticker": "RJF", "name": "Raymond James Financial Inc"},
        {"ticker": "NTRS", "name": "Northern Trust Corp"},
        {"ticker": "L", "name": "Loews Corp"},
        {"ticker": "UNM", "name": "Unum Group"},
        {"ticker": "AIZ", "name": "Assurant Inc"},
        {"ticker": "FDS", "name": "FactSet Research Systems Inc"},
        {"ticker": "MKTX", "name": "MarketAxess Holdings Inc"},
        {"ticker": "NDAQ", "name": "Nasdaq Inc"},
        {"ticker": "CBOE", "name": "Cboe Global Markets Inc"},
        {"ticker": "LPLA", "name": "LPL Financial Holdings Inc"},
        {"ticker": "IVZ", "name": "Invesco Ltd"},
        {"ticker": "CMA", "name": "Comerica Inc"},
        {"ticker": "ZION", "name": "Zions Bancorporation"},
        {"ticker": "ERIE", "name": "Erie Indemnity Co"},
        {"ticker": "JKHY", "name": "Jack Henry & Associates Inc"},
        {"ticker": "BR", "name": "Broadridge Financial Solutions Inc"},
        {"ticker": "MSCI", "name": "MSCI Inc"},
        {"ticker": "BLDR", "name": "Builders FirstSource Inc"},
        {"ticker": "BX", "name": "Blackstone Inc"},

        # Healthcare (65)
        {"ticker": "LLY", "name": "Eli Lilly and Co"},
        {"ticker": "JNJ", "name": "Johnson & Johnson"},
        {"ticker": "UNH", "name": "UnitedHealth Group Inc"},
        {"ticker": "MRK", "name": "Merck & Co Inc"},
        {"ticker": "ABBV", "name": "AbbVie Inc"},
        {"ticker": "PFE", "name": "Pfizer Inc"},
        {"ticker": "TMO", "name": "Thermo Fisher Scientific Inc"},
        {"ticker": "ABT", "name": "Abbott Laboratories"},
        {"ticker": "DHR", "name": "Danaher Corp"},
        {"ticker": "REGN", "name": "Regeneron Pharmaceuticals Inc"},
        {"ticker": "VRTX", "name": "Vertex Pharmaceuticals Inc"},
        {"ticker": "ISRG", "name": "Intuitive Surgical Inc"},
        {"ticker": "BSX", "name": "Boston Scientific Corp"},
        {"ticker": "SYK", "name": "Stryker Corp"},
        {"ticker": "MDT", "name": "Medtronic plc"},
        {"ticker": "CI", "name": "Cigna Group"},
        {"ticker": "BMY", "name": "Bristol-Myers Squibb Co"},
        {"ticker": "GILD", "name": "Gilead Sciences Inc"},
        {"ticker": "AMGN", "name": "Amgen Inc"},
        {"ticker": "CVS", "name": "CVS Health Corp"},
        {"ticker": "ELV", "name": "Elevance Health Inc"},
        {"ticker": "HUM", "name": "Humana Inc"},
        {"ticker": "MRNA", "name": "Moderna Inc"},
        {"ticker": "BIIB", "name": "Biogen Inc"},
        {"ticker": "ZTS", "name": "Zoetis Inc"},
        {"ticker": "IQV", "name": "IQVIA Holdings Inc"},
        {"ticker": "LH", "name": "Labcorp Holdings Inc"},
        {"ticker": "DGX", "name": "Quest Diagnostics Inc"},
        {"ticker": "CAH", "name": "Cardinal Health Inc"},
        {"ticker": "MCK", "name": "McKesson Corp"},
        {"ticker": "COR", "name": "Cencora Inc"},
        {"ticker": "BAX", "name": "Baxter International Inc"},
        {"ticker": "BDX", "name": "Becton Dickinson and Co"},
        {"ticker": "GEHC", "name": "GE HealthCare Technologies Inc"},
        {"ticker": "RMD", "name": "ResMed Inc"},
        {"ticker": "STE", "name": "Steris PLC"},
        {"ticker": "HCA", "name": "HCA Healthcare Inc"},
        {"ticker": "CNC", "name": "Centene Corp"},
        {"ticker": "HOLX", "name": "Hologic Inc"},
        {"ticker": "IDXX", "name": "IDEXX Laboratories Inc"},
        {"ticker": "MTD", "name": "Mettler-Toledo International Inc"},
        {"ticker": "WST", "name": "West Pharmaceutical Services Inc"},
        {"ticker": "EW", "name": "Edwards Lifesciences Corp"},
        {"ticker": "DXCM", "name": "DexCom Inc"},
        {"ticker": "ALGN", "name": "Align Technology Inc"},
        {"ticker": "INCY", "name": "Incyte Corp"},
        {"ticker": "PODD", "name": "Insulet Corp"},
        {"ticker": "MASI", "name": "Masimo Corp"},
        {"ticker": "CRL", "name": "Charles River Laboratories International Inc"},
        {"ticker": "TECH", "name": "Bio-Techne Corp"},
        {"ticker": "MEDP", "name": "Medpace Holdings Inc"},
        {"ticker": "BRKR", "name": "Bruker Corp"},
        {"ticker": "WAT", "name": "Waters Corp"},
        {"ticker": "A", "name": "Agilent Technologies Inc"},
        {"ticker": "RVTY", "name": "Revvity Inc"},
        {"ticker": "COO", "name": "Cooper Cos Inc"},
        {"ticker": "HWM", "name": "Howmet Aerospace Inc"},
        {"ticker": "UHS", "name": "Universal Health Services Inc"},
        {"ticker": "VTRS", "name": "Viatris Inc"},
        {"ticker": "MOH", "name": "Molina Healthcare Inc"},
        {"ticker": "DVA", "name": "DaVita Inc"},
        {"ticker": "XRAY", "name": "Dentsply Sirona Inc"},
        {"ticker": "HSIC", "name": "Henry Schein Inc"},
        {"ticker": "SOLV", "name": "Solventum Corp"},
        {"ticker": "BNTX", "name": "BioNTech SE"},

        # Consumer Discretionary (55)
        {"ticker": "AMZN", "name": "Amazon.com Inc"},
        {"ticker": "TSLA", "name": "Tesla Inc"},
        {"ticker": "HD", "name": "Home Depot Inc"},
        {"ticker": "LOW", "name": "Lowe's Companies Inc"},
        {"ticker": "SBUX", "name": "Starbucks Corp"},
        {"ticker": "MCD", "name": "McDonald's Corp"},
        {"ticker": "BKNG", "name": "Booking Holdings Inc"},
        {"ticker": "TJX", "name": "TJX Companies Inc"},
        {"ticker": "NKE", "name": "Nike Inc"},
        {"ticker": "ABNB", "name": "Airbnb Inc"},
        {"ticker": "CMG", "name": "Chipotle Mexican Grill Inc"},
        {"ticker": "MAR", "name": "Marriott International Inc"},
        {"ticker": "HLT", "name": "Hilton Worldwide Holdings Inc"},
        {"ticker": "RCL", "name": "Royal Caribbean Cruises Ltd"},
        {"ticker": "CCL", "name": "Carnival Corp"},
        {"ticker": "DHI", "name": "DR Horton Inc"},
        {"ticker": "LEN", "name": "Lennar Corp"},
        {"ticker": "NVR", "name": "NVR Inc"},
        {"ticker": "PHM", "name": "PulteGroup Inc"},
        {"ticker": "TOL", "name": "Toll Brothers Inc"},
        {"ticker": "F", "name": "Ford Motor Co"},
        {"ticker": "GM", "name": "General Motors Co"},
        {"ticker": "APTV", "name": "Aptiv PLC"},
        {"ticker": "BWA", "name": "BorgWarner Inc"},
        {"ticker": "LKQ", "name": "LKQ Corp"},
        {"ticker": "GPC", "name": "Genuine Parts Co"},
        {"ticker": "DRI", "name": "Darden Restaurants Inc"},
        {"ticker": "YUM", "name": "Yum! Brands Inc"},
        {"ticker": "DPZ", "name": "Domino's Pizza Inc"},
        {"ticker": "QSR", "name": "Restaurant Brands International Inc"},
        {"ticker": "TSCO", "name": "Tractor Supply Co"},
        {"ticker": "ULTA", "name": "Ulta Beauty Inc"},
        {"ticker": "DG", "name": "Dollar General Corp"},
        {"ticker": "DLTR", "name": "Dollar Tree Inc"},
        {"ticker": "ROST", "name": "Ross Stores Inc"},
        {"ticker": "EBAY", "name": "eBay Inc"},
        {"ticker": "AZO", "name": "AutoZone Inc"},
        {"ticker": "ORLY", "name": "O'Reilly Automotive Inc"},
        {"ticker": "BBY", "name": "Best Buy Co Inc"},
        {"ticker": "POOL", "name": "Pool Corp"},
        {"ticker": "HAS", "name": "Hasbro Inc"},
        {"ticker": "MAT", "name": "Mattel Inc"},
        {"ticker": "ETSY", "name": "Etsy Inc"},
        {"ticker": "CVNA", "name": "Carvana Co"},
        {"ticker": "WYNN", "name": "Wynn Resorts Ltd"},
        {"ticker": "LVS", "name": "Las Vegas Sands Corp"},
        {"ticker": "MGM", "name": "MGM Resorts International"},
        {"ticker": "CZR", "name": "Caesars Entertainment Inc"},
        {"ticker": "TPR", "name": "Tapestry Inc"},
        {"ticker": "PVH", "name": "PVH Corp"},
        {"ticker": "RL", "name": "Ralph Lauren Corp"},
        {"ticker": "DECK", "name": "Deckers Outdoor Corp"},
        {"ticker": "GRMN", "name": "Garmin Ltd"},
        {"ticker": "LULU", "name": "Lululemon Athletica Inc"},
        {"ticker": "PHM", "name": "PulteGroup Inc"},

        # Communication Services (24)
        {"ticker": "META", "name": "Meta Platforms Inc"},
        {"ticker": "GOOGL", "name": "Alphabet Inc Class A"},
        {"ticker": "GOOG", "name": "Alphabet Inc Class C"},
        {"ticker": "NFLX", "name": "Netflix Inc"},
        {"ticker": "DIS", "name": "Walt Disney Co"},
        {"ticker": "CMCSA", "name": "Comcast Corp"},
        {"ticker": "T", "name": "AT&T Inc"},
        {"ticker": "VZ", "name": "Verizon Communications Inc"},
        {"ticker": "TMUS", "name": "T-Mobile US Inc"},
        {"ticker": "EA", "name": "Electronic Arts Inc"},
        {"ticker": "TTWO", "name": "Take-Two Interactive Software Inc"},
        {"ticker": "WBD", "name": "Warner Bros Discovery Inc"},
        {"ticker": "PARA", "name": "Paramount Global"},
        {"ticker": "FOXA", "name": "Fox Corp Class A"},
        {"ticker": "FOX", "name": "Fox Corp Class B"},
        {"ticker": "NWSA", "name": "News Corp Class A"},
        {"ticker": "NWS", "name": "News Corp Class B"},
        {"ticker": "LYV", "name": "Live Nation Entertainment Inc"},
        {"ticker": "CHTR", "name": "Charter Communications Inc"},
        {"ticker": "OMC", "name": "Omnicom Group Inc"},
        {"ticker": "IPG", "name": "Interpublic Group of Companies Inc"},
        {"ticker": "NYT", "name": "New York Times Co"},
        {"ticker": "MTCH", "name": "Match Group Inc"},
        {"ticker": "EDR", "name": "Endeavor Group Holdings Inc"},

        # Consumer Staples (38)
        {"ticker": "WMT", "name": "Walmart Inc"},
        {"ticker": "COST", "name": "Costco Wholesale Corp"},
        {"ticker": "PG", "name": "Procter & Gamble Co"},
        {"ticker": "KO", "name": "Coca-Cola Co"},
        {"ticker": "PEP", "name": "PepsiCo Inc"},
        {"ticker": "PM", "name": "Philip Morris International Inc"},
        {"ticker": "MO", "name": "Altria Group Inc"},
        {"ticker": "MDLZ", "name": "Mondelez International Inc"},
        {"ticker": "CL", "name": "Colgate-Palmolive Co"},
        {"ticker": "KMB", "name": "Kimberly-Clark Corp"},
        {"ticker": "KHC", "name": "Kraft Heinz Co"},
        {"ticker": "GIS", "name": "General Mills Inc"},
        {"ticker": "K", "name": "Kellanova"},
        {"ticker": "CPB", "name": "Campbell Soup Co"},
        {"ticker": "CAG", "name": "Conagra Brands Inc"},
        {"ticker": "SJM", "name": "JM Smucker Co"},
        {"ticker": "HRL", "name": "Hormel Foods Corp"},
        {"ticker": "TSN", "name": "Tyson Foods Inc"},
        {"ticker": "ADM", "name": "Archer-Daniels-Midland Co"},
        {"ticker": "BG", "name": "Bunge Global SA"},
        {"ticker": "STZ", "name": "Constellation Brands Inc"},
        {"ticker": "BF.B", "name": "Brown-Forman Corp"},
        {"ticker": "MNST", "name": "Monster Beverage Corp"},
        {"ticker": "KVUE", "name": "Kenvue Inc"},
        {"ticker": "CHD", "name": "Church & Dwight Co Inc"},
        {"ticker": "CLX", "name": "Clorox Co"},
        {"ticker": "EL", "name": "Estee Lauder Companies Inc"},
        {"ticker": "SYY", "name": "Sysco Corp"},
        {"ticker": "USFD", "name": "US Foods Holding Corp"},
        {"ticker": "PFGC", "name": "Performance Food Group Co"},
        {"ticker": "LW", "name": "Lamb Weston Holdings Inc"},
        {"ticker": "TAP", "name": "Molson Coors Beverage Co"},
        {"ticker": "MKC", "name": "McCormick & Co Inc"},
        {"ticker": "HBI", "name": "Hanesbrands Inc"},
        {"ticker": "KR", "name": "Kroger Co"},
        {"ticker": "WBA", "name": "Walgreens Boots Alliance Inc"},
        {"ticker": "DLTR", "name": "Dollar Tree Inc"},
        {"ticker": "TGT", "name": "Target Corp"},

        # Energy (22)
        {"ticker": "XOM", "name": "Exxon Mobil Corp"},
        {"ticker": "CVX", "name": "Chevron Corp"},
        {"ticker": "COP", "name": "ConocoPhillips"},
        {"ticker": "EOG", "name": "EOG Resources Inc"},
        {"ticker": "SLB", "name": "Schlumberger NV"},
        {"ticker": "OXY", "name": "Occidental Petroleum Corp"},
        {"ticker": "PSX", "name": "Phillips 66"},
        {"ticker": "VLO", "name": "Valero Energy Corp"},
        {"ticker": "MPC", "name": "Marathon Petroleum Corp"},
        {"ticker": "HES", "name": "Hess Corp"},
        {"ticker": "DVN", "name": "Devon Energy Corp"},
        {"ticker": "FANG", "name": "Diamondback Energy Inc"},
        {"ticker": "CTRA", "name": "Coterra Energy Inc"},
        {"ticker": "EQT", "name": "EQT Corp"},
        {"ticker": "KMI", "name": "Kinder Morgan Inc"},
        {"ticker": "WMB", "name": "Williams Companies Inc"},
        {"ticker": "OKE", "name": "ONEOK Inc"},
        {"ticker": "TRGP", "name": "Targa Resources Corp"},
        {"ticker": "BKR", "name": "Baker Hughes Co"},
        {"ticker": "HAL", "name": "Halliburton Co"},
        {"ticker": "APA", "name": "APA Corp"},
        {"ticker": "MRO", "name": "Marathon Oil Corp"},

        # Industrials (78)
        {"ticker": "GE", "name": "General Electric Co"},
        {"ticker": "CAT", "name": "Caterpillar Inc"},
        {"ticker": "BA", "name": "Boeing Co"},
        {"ticker": "UPS", "name": "United Parcel Service Inc"},
        {"ticker": "RTX", "name": "RTX Corp"},
        {"ticker": "HON", "name": "Honeywell International Inc"},
        {"ticker": "UNP", "name": "Union Pacific Corp"},
        {"ticker": "LMT", "name": "Lockheed Martin Corp"},
        {"ticker": "DE", "name": "Deere & Co"},
        {"ticker": "FDX", "name": "FedEx Corp"},
        {"ticker": "ITW", "name": "Illinois Tool Works Inc"},
        {"ticker": "MMM", "name": "3M Co"},
        {"ticker": "EMR", "name": "Emerson Electric Co"},
        {"ticker": "ETN", "name": "Eaton Corp"},
        {"ticker": "NOC", "name": "Northrop Grumman Corp"},
        {"ticker": "GD", "name": "General Dynamics Corp"},
        {"ticker": "PH", "name": "Parker-Hannifin Corp"},
        {"ticker": "CMI", "name": "Cummins Inc"},
        {"ticker": "PCAR", "name": "PACCAR Inc"},
        {"ticker": "CSX", "name": "CSX Corp"},
        {"ticker": "NSC", "name": "Norfolk Southern Corp"},
        {"ticker": "ODFL", "name": "Old Dominion Freight Line Inc"},
        {"ticker": "JBHT", "name": "JB Hunt Transport Services Inc"},
        {"ticker": "CHRW", "name": "CH Robinson Worldwide Inc"},
        {"ticker": "EXPD", "name": "Expeditors International of Washington Inc"},
        {"ticker": "LUV", "name": "Southwest Airlines Co"},
        {"ticker": "DAL", "name": "Delta Air Lines Inc"},
        {"ticker": "UAL", "name": "United Airlines Holdings Inc"},
        {"ticker": "AAL", "name": "American Airlines Group Inc"},
        {"ticker": "GWW", "name": "WW Grainger Inc"},
        {"ticker": "FAST", "name": "Fastenal Co"},
        {"ticker": "PAYX", "name": "Paychex Inc"},
        {"ticker": "ADP", "name": "Automatic Data Processing Inc"},
        {"ticker": "CTAS", "name": "Cintas Corp"},
        {"ticker": "RSG", "name": "Republic Services Inc"},
        {"ticker": "WM", "name": "Waste Management Inc"},
        {"ticker": "XPO", "name": "XPO Logistics Inc"},
        {"ticker": "LSTR", "name": "Landstar System Inc"},
        {"ticker": "HII", "name": "Huntington Ingalls Industries Inc"},
        {"ticker": "TXT", "name": "Textron Inc"},
        {"ticker": "TDY", "name": "Teledyne Technologies Inc"},
        {"ticker": "AME", "name": "Ametek Inc"},
        {"ticker": "IR", "name": "Ingersoll Rand Inc"},
        {"ticker": "JCI", "name": "Johnson Controls International PLC"},
        {"ticker": "TT", "name": "Trane Technologies PLC"},
        {"ticker": "CARR", "name": "Carrier Global Corp"},
        {"ticker": "OTIS", "name": "Otis Worldwide Corp"},
        {"ticker": "ALLE", "name": "Allegion PLC"},
        {"ticker": "AOS", "name": "Smith (AO) Corp"},
        {"ticker": "LDOS", "name": "Leidos Holdings Inc"},
        {"ticker": "SAIC", "name": "Science Applications International Corp"},
        {"ticker": "ROL", "name": "Rollins Inc"},
        {"ticker": "ROK", "name": "Rockwell Automation Inc"},
        {"ticker": "DOV", "name": "Dover Corp"},
        {"ticker": "SWK", "name": "Stanley Black & Decker Inc"},
        {"ticker": "SNA", "name": "Snap-on Inc"},
        {"ticker": "TTC", "name": "Toro Co"},
        {"ticker": "GGG", "name": "Graco Inc"},
        {"ticker": "NDSN", "name": "Nordson Corp"},
        {"ticker": "RHI", "name": "Robert Half Inc"},
        {"ticker": "MAN", "name": "ManpowerGroup Inc"},
        {"ticker": "J", "name": "Jacobs Solutions Inc"},
        {"ticker": "EME", "name": "EMCOR Group Inc"},
        {"ticker": "ACM", "name": "AECOM"},
        {"ticker": "PNR", "name": "Pentair PLC"},
        {"ticker": "IEX", "name": "IDEX Corp"},
        {"ticker": "WAB", "name": "Westinghouse Air Brake Technologies Corp"},
        {"ticker": "XYL", "name": "Xylem Inc"},
        {"ticker": "GEV", "name": "GE Vernova Inc"},
        {"ticker": "GNRC", "name": "Generac Holdings Inc"},
        {"ticker": "VMC", "name": "Vulcan Materials Co"},
        {"ticker": "MLM", "name": "Martin Marietta Materials Inc"},
        {"ticker": "EXP", "name": "Eagle Materials Inc"},
        {"ticker": "LNX", "name": "Lennox International Inc"},
        {"ticker": "FIX", "name": "Comfort Systems USA Inc"},
        {"ticker": "URI", "name": "United Rentals Inc"},
        {"ticker": "AXON", "name": "Axon Enterprise Inc"},
        {"ticker": "HUBB", "name": "Hubbell Inc"},

        # Materials (25)
        {"ticker": "LIN", "name": "Linde PLC"},
        {"ticker": "APD", "name": "Air Products and Chemicals Inc"},
        {"ticker": "ECL", "name": "Ecolab Inc"},
        {"ticker": "SHW", "name": "Sherwin-Williams Co"},
        {"ticker": "PPG", "name": "PPG Industries Inc"},
        {"ticker": "NEM", "name": "Newmont Corp"},
        {"ticker": "FCX", "name": "Freeport-McMoRan Inc"},
        {"ticker": "NUE", "name": "Nucor Corp"},
        {"ticker": "STLD", "name": "Steel Dynamics Inc"},
        {"ticker": "RS", "name": "Reliance Steel & Aluminum Co"},
        {"ticker": "DOW", "name": "Dow Inc"},
        {"ticker": "DD", "name": "DuPont de Nemours Inc"},
        {"ticker": "LYB", "name": "LyondellBasell Industries NV"},
        {"ticker": "WLK", "name": "Westlake Corp"},
        {"ticker": "CE", "name": "Celanese Corp"},
        {"ticker": "EMN", "name": "Eastman Chemical Co"},
        {"ticker": "ALB", "name": "Albemarle Corp"},
        {"ticker": "IFF", "name": "International Flavors & Fragrances Inc"},
        {"ticker": "CTVA", "name": "Corteva Inc"},
        {"ticker": "FMC", "name": "FMC Corp"},
        {"ticker": "MOS", "name": "Mosaic Co"},
        {"ticker": "CF", "name": "CF Industries Holdings Inc"},
        {"ticker": "PKG", "name": "Packaging Corp of America"},
        {"ticker": "WRK", "name": "WestRock Co"},
        {"ticker": "IP", "name": "International Paper Co"},

        # Real Estate (31)
        {"ticker": "PLD", "name": "Prologis Inc"},
        {"ticker": "EQIX", "name": "Equinix Inc"},
        {"ticker": "AMT", "name": "American Tower Corp"},
        {"ticker": "DLR", "name": "Digital Realty Trust Inc"},
        {"ticker": "CCI", "name": "Crown Castle International Corp"},
        {"ticker": "PSA", "name": "Public Storage"},
        {"ticker": "WELL", "name": "Welltower Inc"},
        {"ticker": "AVB", "name": "AvalonBay Communities Inc"},
        {"ticker": "EQR", "name": "Equity Residential"},
        {"ticker": "ESS", "name": "Essex Property Trust Inc"},
        {"ticker": "MAA", "name": "Mid-America Apartment Communities Inc"},
        {"ticker": "INVH", "name": "Invitation Homes Inc"},
        {"ticker": "SUI", "name": "Sun Communities Inc"},
        {"ticker": "UDR", "name": "UDR Inc"},
        {"ticker": "VTR", "name": "Ventas Inc"},
        {"ticker": "ARE", "name": "Alexandria Real Estate Equities Inc"},
        {"ticker": "BXP", "name": "Boston Properties Inc"},
        {"ticker": "SPG", "name": "Simon Property Group Inc"},
        {"ticker": "KIM", "name": "Kimco Realty Corp"},
        {"ticker": "REG", "name": "Regency Centers Corp"},
        {"ticker": "FRT", "name": "Federal Realty Investment Trust"},
        {"ticker": "O", "name": "Realty Income Corp"},
        {"ticker": "IRM", "name": "Iron Mountain Inc"},
        {"ticker": "WY", "name": "Weyerhaeuser Co"},
        {"ticker": "HST", "name": "Host Hotels & Resorts Inc"},
        {"ticker": "PEAK", "name": "Healthpeak Properties Inc"},
        {"ticker": "DOC", "name": "Physicians Realty Trust"},
        {"ticker": "CUZ", "name": "Cousins Properties Inc"},
        {"ticker": "KRC", "name": "Kilroy Realty Corp"},
        {"ticker": "SLG", "name": "SL Green Realty Corp"},
        {"ticker": "VICI", "name": "VICI Properties Inc"},

        # Utilities (31)
        {"ticker": "NEE", "name": "NextEra Energy Inc"},
        {"ticker": "DUK", "name": "Duke Energy Corp"},
        {"ticker": "SO", "name": "Southern Co"},
        {"ticker": "AEP", "name": "American Electric Power Co Inc"},
        {"ticker": "EXC", "name": "Exelon Corp"},
        {"ticker": "SRE", "name": "Sempra"},
        {"ticker": "PEG", "name": "Public Service Enterprise Group Inc"},
        {"ticker": "XEL", "name": "Xcel Energy Inc"},
        {"ticker": "ED", "name": "Consolidated Edison Inc"},
        {"ticker": "EIX", "name": "Edison International"},
        {"ticker": "WEC", "name": "WEC Energy Group Inc"},
        {"ticker": "PPL", "name": "PPL Corp"},
        {"ticker": "DTE", "name": "DTE Energy Co"},
        {"ticker": "ETR", "name": "Entergy Corp"},
        {"ticker": "FE", "name": "FirstEnergy Corp"},
        {"ticker": "AEE", "name": "Ameren Corp"},
        {"ticker": "CMS", "name": "CMS Energy Corp"},
        {"ticker": "CNP", "name": "CenterPoint Energy Inc"},
        {"ticker": "LNT", "name": "Alliant Energy Corp"},
        {"ticker": "NI", "name": "NiSource Inc"},
        {"ticker": "EVRG", "name": "Evergy Inc"},
        {"ticker": "OGE", "name": "OGE Energy Corp"},
        {"ticker": "ATO", "name": "Atmos Energy Corp"},
        {"ticker": "AES", "name": "AES Corp"},
        {"ticker": "NRG", "name": "NRG Energy Inc"},
        {"ticker": "VST", "name": "Vistra Corp"},
        {"ticker": "CEG", "name": "Constellation Energy Corp"},
        {"ticker": "UGI", "name": "UGI Corp"},
        {"ticker": "NFG", "name": "National Fuel Gas Co"},
        {"ticker": "PNW", "name": "Pinnacle West Capital Corp"},
        {"ticker": "AWK", "name": "American Water Works Co Inc"},
    ]

# To verify count:
# print(len(get_sp500_tickers())) # Output: 500

In [4]:
"""
SEC Fine-Tuning Dataset Builder  v13  — RESEARCH-GRADE REASONING DATASET
=========================================================================
Target score: 97-98+ / 100

EXPERT AUDIT IMPLEMENTATION (all 12 upgrade areas from review):
────────────────────────────────────────────────────────────────
1.  HARD CAUSALITY CONSTRAINTS       — causal_validator() + CausalGuard template engine
2.  DEEP MD&A INTEGRATION            — MDAGrounder: semantic alignment text↔metric changes
3.  LOW-SIGNAL LANGUAGE REMOVAL      — StyleCompressor + banned_token_filter()
4.  QUANTIFY EVERYTHING              — strict magnitude enforcement in prompts + output validation
5.  EDGE-CASE REASONING TEMPLATES    — EdgeCaseClassifier + 5 contradiction archetypes
6.  CONFIDENCE-AWARE OUTPUTS         — EvidenceConfidence scoring in every output
7.  NORMALIZE FINANCIAL SEMANTICS    — CANONICAL_TERMS dictionary + normalization pass
8.  REMOVE TEMPLATE DETECTABILITY    — StructuralRandomizer (4 output orderings)
9.  NEGATIVE REASONING EXAMPLES      — NegativeExampleBuilder (what CANNOT be concluded)
10. MULTI-HORIZON REASONING          — QoQ + YoY + 3yr CAGR in structured horizon block
11. SUPERVISION TYPE LABELS          — reasoning_type metadata tags for curriculum learning
12. QUALITY VALIDATION PIPELINE      — QualityValidator: hallucination + delta + style + diversity

ARCHITECTURE:
  ┌─────────────────────────────────────────────────────────────────────────┐
  │  EDGAR → XBRL metrics → CausalGuard → MDAGrounder → EdgeClassifier     │
  │  → ConfidenceScorer → PromptBuilder → LLM → QualityValidator → JSONL   │
  └─────────────────────────────────────────────────────────────────────────┘

LLM: KIMI K2 (unlimited TPD) → Groq fallback
INSTALL: pip install requests beautifulsoup4 lxml loguru langchain-groq langchain-openai python-dotenv
"""

import os, re, json, sys, time, hashlib, random, warnings
from collections import defaultdict
from datetime import datetime, timedelta
from typing import Optional

import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from loguru import logger
from tqdm import tqdm

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# ─── LOGURU ───────────────────────────────────────────────────────────────────
logger.remove()
logger.add(sys.stderr,
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <7}</level> | {message}",
    level="INFO", colorize=True)
logger.add("sec_builder_v13.log",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <7} | {function}:{line} | {message}",
    level="DEBUG", rotation="50 MB", retention="7 days")





# ─── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_FILE             = "sec_finetune_v13.jsonl"
PENDING_FILE            = "sec_pending_v13.jsonl"
QUALITY_LOG_FILE        = "sec_quality_v13.jsonl"
YEARS_BACK              = 5
EDGAR_SLEEP             = 0.15
MAX_RETRIES             = 4
BACKOFF_BASE            = 2.0
PERIOD_FUZZ_DAYS        = 5
MAX_TEXT_CHARS          = 4500
MIN_PARA_LEN            = 60
MIN_PARA_ALPHA          = 0.50
LLM_429_MAX_WAIT        = 400
MIN_NARRATIVE_DOC_BYTES = 200_000

EDGAR_HEADERS = {
    "User-Agent": "SECDatasetBuilder/1.0 (ML research; research@university.edu)",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "application/json, text/html, */*",
}

# ─── UPGRADE #7: CANONICAL FINANCIAL TERMINOLOGY ──────────────────────────────
CANONICAL_TERMS = {
    # Non-canonical → canonical
    "profitability pressure":   "margin pressure",
    "earnings pressure":        "margin pressure",
    "profit decline":           "margin pressure",
    "profit compression":       "margin pressure",
    "margin deterioration":     "margin pressure",
    "debt burden":              "leverage pressure",
    "debt load":                "leverage pressure",
    "heavy debt":               "leverage pressure",
    "liquidity issue":          "liquidity pressure",
    "liquidity concern":        "liquidity pressure",
    "cash crunch":              "liquidity pressure",
    "revenue contraction":      "revenue decline",
    "top-line growth":          "revenue growth",
    "top-line decline":         "revenue decline",
    "bottom-line growth":       "net income growth",
    "bottom-line pressure":     "margin pressure",
    "operational efficiency":   "operating leverage improvement",
    "cost optimization":        "cost structure improvement",
    "capital efficiency":       "asset utilization",
    "strong performance":       "above-average financial results",
    "weak performance":         "below-average financial results",
    "showing strength":         "metrics above historical average",
    "showing weakness":         "metrics below historical average",
}

# ─── UPGRADE #3: LOW-SIGNAL TOKEN BLACKLIST ────────────────────────────────────
FILLER_TOKENS = {
    "suggesting", "indicating", "overall", "reflects", "demonstrates",
    "showcases", "underscores", "highlights", "illustrates", "notably",
    "importantly", "significantly", "meaningfully", "materially",
    "generally", "broadly", "largely", "essentially", "fundamentally",
    "incredibly", "remarkably", "substantially", "considerably",
    "improved", "declined", "strong", "weak",  # only when no magnitude follows
    "robust", "solid", "healthy", "challenging", "difficult",
}

# ─── UPGRADE #9: NEGATIVE REASONING CORPUS ────────────────────────────────────
# Evidence thresholds for supported vs unsupported causal inference
NEGATIVE_REASONING_TEMPLATES = [
    {
        "trigger": "revenue_yoy_pct < 5 and revenue_yoy_pct > 0",
        "bad_conclusion": "Strong market demand drove revenue.",
        "correct_conclusion": "Revenue grew modestly at {revenue_yoy_pct:.1f}% YoY; no strong acceleration signal. No explicit growth driver identifiable from available data.",
        "label": "weak_growth_overclaim"
    },
    {
        "trigger": "revenue_yoy_pct > 15 and not mda_has_text",
        "bad_conclusion": "Product demand and market expansion drove growth.",
        "correct_conclusion": "Revenue grew {revenue_yoy_pct:.1f}% YoY. No MD&A text available; growth driver unidentifiable — quantitative signal only.",
        "label": "growth_without_driver_evidence"
    },
    {
        "trigger": "gross_margin_pct_delta < -2 and not mda_has_text",
        "bad_conclusion": "Supply chain disruption caused margin compression.",
        "correct_conclusion": "Gross margin compressed {gross_margin_pct_delta:.1f}pp YoY. No explicit cost driver identified in available data.",
        "label": "margin_compression_unsupported"
    },
    {
        "trigger": "operating_margin_pct > 20 and net_margin_pct < 5",
        "bad_conclusion": "Strong operations indicate overall business health.",
        "correct_conclusion": "Operating margin at {operating_margin_pct:.1f}% but net margin only {net_margin_pct:.1f}% — significant below-the-line costs (interest, tax, or non-operating) materially reduce profitability. High operating margin does NOT imply overall financial health.",
        "label": "operating_vs_net_margin_disconnect"
    },
    {
        "trigger": "fcf_yoy_pct > 0 and revenue_yoy_pct < -5",
        "bad_conclusion": "Business momentum is positive given improving cash flows.",
        "correct_conclusion": "FCF improved {fcf_yoy_pct:.1f}% YoY despite revenue declining {revenue_yoy_pct:.1f}%. FCF improvement during revenue decline may reflect cost-cutting or CapEx reduction, not business acceleration. Cannot conclude positive momentum.",
        "label": "fcf_improvement_revenue_decline"
    },
]

# ─── UPGRADE #5: EDGE CASE ARCHETYPES ─────────────────────────────────────────
EDGE_CASE_ARCHETYPES = {
    "unprofitable_expansion": {
        "condition": lambda m, c: (c.get("revenue_yoy_pct", 0) or 0) > 15 and (m.get("operating_margin_pct", 0) or 0) < -5,
        "label": "growth_margin_collapse",
        "reasoning_focus": "Distinguish revenue scale from profitability. Teach: high growth + negative margins = investment phase or structural cost problem.",
    },
    "efficiency_harvest": {
        "condition": lambda m, c: abs((c.get("revenue_yoy_pct", 0) or 0)) < 3 and (c.get("fcf_yoy_pct", 0) or 0) > 15,
        "label": "flat_revenue_fcf_improvement",
        "reasoning_focus": "Teach: flat revenue + improving FCF = operational efficiency or CapEx reduction, not growth stagnation.",
    },
    "restructuring_signal": {
        "condition": lambda m, c: (c.get("revenue_yoy_pct", 0) or 0) < -5 and (c.get("gross_margin_pct_delta", 0) or 0) > 2,
        "label": "revenue_decline_margin_improvement",
        "reasoning_focus": "Teach: revenue decline + margin improvement = portfolio pruning, restructuring, or mix shift toward premium products.",
    },
    "manageable_leverage": {
        "condition": lambda m, c: (m.get("debt_to_equity", 0) or 0) > 2.0 and (c.get("fcf_yoy_pct", 0) or 0) > 0 and (m.get("free_cash_flow", 0) or 0) > 0,
        "label": "high_leverage_strong_cashflow",
        "reasoning_focus": "Teach: high D/E ratio alone is insufficient for leverage risk assessment — FCF coverage determines debt serviceability.",
    },
    "liquidity_paradox": {
        "condition": lambda m, c: (m.get("current_ratio", 99) or 99) < 1.0 and (m.get("cash_and_equivalents", 0) or 0) > 1e9,
        "label": "low_current_ratio_high_cash",
        "reasoning_focus": "Teach: current ratio below 1.0 with large absolute cash balance — nuanced interpretation required, not automatic liquidity crisis.",
    },
    "conflicting_earnings": {
        "condition": lambda m, c: (c.get("revenue_yoy_pct", 0) or 0) > 10 and (c.get("net_income_yoy_pct", 0) or 0) < -10,
        "label": "revenue_growth_income_decline",
        "reasoning_focus": "Teach: revenue growth with net income decline = cost structure problem, investment cycle, or tax/interest event. Cannot conclude business health from revenue alone.",
    },
}

# ─── UPGRADE #11: REASONING TYPE TAXONOMY ─────────────────────────────────────
REASONING_TYPES = {
    "trend_analysis":       lambda m, c: c.get("revenue_yoy_pct") is not None,
    "margin_analysis":      lambda m, c: m.get("gross_margin_pct") is not None and m.get("net_margin_pct") is not None,
    "risk_assessment":      lambda m, c: m.get("debt_to_equity") is not None or m.get("current_ratio") is not None,
    "cash_flow_quality":    lambda m, c: m.get("free_cash_flow") is not None and m.get("operating_cash_flow") is not None,
    "capital_allocation":   lambda m, c: m.get("capex") is not None and m.get("share_repurchases") is not None,
    "profitability_driver": lambda m, c: c.get("gross_margin_pct_delta") is not None,
    "leverage_assessment":  lambda m, c: m.get("long_term_debt") is not None and m.get("net_debt") is not None,
    "growth_quality":       lambda m, c: c.get("revenue_yoy_pct") is not None and c.get("eps_yoy_pct") is not None,
    "multi_horizon":        lambda m, c: False,  # set externally for multi-year records
    "negative_inference":   lambda m, c: False,  # set externally for negative examples
    "edge_case":            lambda m, c: False,  # set externally for edge cases
}

FINANCIAL_KEYWORDS = {
    "revenue","net income","earnings","operating","margin","profit",
    "growth","decline","increased","decreased","compared","fiscal",
    "quarter","year","segment","billion","million","percent","%",
    "cash flow","capex","ebitda","gross","cost","expense","loss",
    "gain","acquisition","guidance","outlook","demand","supply",
    "customers","market share","competition",
}

MATERIAL_8K_KEYWORDS = {
    "revenue","earnings","net income","quarterly results","annual results",
    "acquisition","merger","acquires","acquired","divest",
    "ceo","cfo","chief executive","chief financial","resign","appoint",
    "restructur","layoff","workforce reduction",
    "lawsuit","litigation","settlement","investigation",
    "guidance","outlook","dividend","buyback","repurchase","impairment",
}

ITEM_HEADING_RE = re.compile(r"\bitem\s+(?:1a?|2|3|7a?|8)\b", re.IGNORECASE)

SECTIONS_10K = {
    "business":             [r"item\s+1[\.\s]*[-–—]?\s*business\b",           r"(?<!\w)item\s+1(?!\s*[aAbB])\b"],
    "risk_factors":         [r"item\s+1a[\.\s]*[-–—]?\s*risk\s+factor",       r"item\s+1a\b"],
    "mda":                  [r"item\s+7[\.\s]*[-–—]?\s*management.{0,120}discussion", r"(?<!\w)item\s+7(?!\s*a)\b"],
    "market_risk":          [r"item\s+7a[\.\s]*[-–—]?\s*quantitative",        r"item\s+7a\b"],
    "financial_statements": [r"item\s+8[\.\s]*[-–—]?\s*financial\s+statement", r"(?<!\w)item\s+8\b"],
    "legal_proceedings":    [r"item\s+3[\.\s]*[-–—]?\s*legal\s+proceed",      r"(?<!\w)item\s+3\b"],
}
SECTIONS_10Q = {
    "mda":                  [r"item\s+2[\.\s]*[-–—]?\s*management.{0,120}discussion", r"item\s+2\b"],
    "market_risk":          [r"item\s+3[\.\s]*[-–—]?\s*quantitative",         r"item\s+3\b"],
    "risk_factors":         [r"item\s+1a[\.\s]*[-–—]?\s*risk\s+factor",       r"item\s+1a\b"],
    "financial_statements": [r"item\s+1[\.\s]*[-–—]?\s*financial\s+statement", r"(?<!\w)item\s+1(?!\s*a)\b"],
    "legal_proceedings":    [r"item\s+1[\.\s]*[-–—]?\s*legal\s+proceed"],
}

NEXT_ITEM_RE = re.compile(r"\bitem\s+\d{1,2}[ab]?\s*[\.\s]", re.IGNORECASE)
_NUM_RE      = re.compile(r"(\$[\d,.]+[BbMmKk]?|\d+\.?\d*\s*%|\b\d{1,3}(?:,\d{3})+\b)", re.IGNORECASE)

RISK_KW = [
    "competition","regulation","supply chain","cybersecurity","inflation",
    "interest rate","liquidity","litigation","currency risk","macroeconomic",
    "recession","geopolitical","tariff","sanctions","climate","data privacy",
    "customer concentration","patent","labor shortage","ai disruption",
]


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE #1: CAUSAL GUARD — Hard Causality Constraint Engine
# ══════════════════════════════════════════════════════════════════════════════

class CausalGuard:
    """
    Enforces evidence-grounded causality in LLM prompts and output validation.
    Prevents hallucinated product/segment causal attribution without MD&A support.
    """

    # Patterns that signal unsupported causal attribution
    UNSUPPORTED_CAUSAL_PATTERNS = [
        r"\b(iphone|ipad|mac|airpods|services)\b.*\b(drove|driven|boosted|caused|led to)\b",
        r"\b(drove|driven|boosted|caused|led to)\b.*\b(revenue|growth|sales)\b",
        r"\bdue to (product|segment|customer|market) (demand|adoption|growth)\b",
        r"\b(strong|robust|high) demand (drove|boosted|caused)\b",
        r"\boperational efficiency (drove|caused|led to) (revenue|margin|profit)\b",
    ]
    _compiled = [re.compile(p, re.IGNORECASE) for p in UNSUPPORTED_CAUSAL_PATTERNS]

    MDA_DRIVER_PATTERNS = {
        "cost":         [r"cost(s)?\s+(increased|higher|rose|surge)", r"input cost", r"logistics cost", r"labor cost"],
        "demand":       [r"demand (increased|higher|strong|grew)", r"customer (demand|volume)", r"order volume"],
        "pricing":      [r"price (increase|increase|higher)", r"pricing power", r"ASP"],
        "volume":       [r"unit (volume|sales|shipped)", r"shipment(s)?", r"units sold"],
        "fx":           [r"foreign (exchange|currency)", r"currency (headwind|tailwind|impact)"],
        "acquisition":  [r"acqui(red|sition|re)", r"merger", r"inorganic"],
        "restructuring":[r"restructur", r"headcount reduction", r"cost reduction program"],
        "macro":        [r"macroeconomic", r"recession", r"inflation", r"interest rate"],
    }
    _mda_compiled = {k: [re.compile(p, re.IGNORECASE) for p in pats]
                     for k, pats in MDA_DRIVER_PATTERNS.items()}

    @classmethod
    def detect_drivers_in_mda(cls, mda_text: str) -> dict[str, bool]:
        """Extract which causal drivers have explicit MD&A evidence."""
        if not mda_text:
            return {}
        found = {}
        for driver, patterns in cls._mda_compiled.items():
            found[driver] = any(p.search(mda_text) for p in patterns)
        return found

    @classmethod
    def validate_output(cls, output_text: str, mda_text: str) -> tuple[bool, list[str]]:
        """
        Returns (is_valid, list_of_violations).
        Checks for unsupported causal statements when no MD&A evidence exists.
        """
        if mda_text and len(mda_text) > 200:
            return True, []  # MD&A available — allow grounded claims
        violations = []
        for pattern in cls._compiled:
            m = pattern.search(output_text)
            if m:
                violations.append(f"Unsupported causal claim: '{m.group()[:80]}'")
        return len(violations) == 0, violations

    @classmethod
    def build_causality_instruction(cls, metrics: dict, changes: dict, mda_text: str) -> str:
        """Build precise causality instruction based on evidence availability."""
        has_mda = bool(mda_text and len(mda_text) > 200)
        drivers = cls.detect_drivers_in_mda(mda_text) if has_mda else {}

        if has_mda and drivers:
            allowed = [k for k, v in drivers.items() if v]
            return (
                f"CAUSALITY RULE: MD&A text is available. You may attribute causal drivers "
                f"ONLY to the following evidence-backed themes found in MD&A: {', '.join(allowed)}. "
                f"Do NOT invent product-level attribution (e.g. 'iPhone sales') unless the exact "
                f"product/segment name appears verbatim in the MD&A excerpt below."
            )
        elif has_mda and not drivers:
            return (
                "CAUSALITY RULE: MD&A available but no clear cost/demand/pricing drivers detected. "
                "Use metric-level causality only (e.g. 'gross margin compressed X pp', 'R&D intensity at X%'). "
                "Do NOT fabricate narrative drivers."
            )
        else:
            return (
                "CAUSALITY RULE: NO MD&A TEXT AVAILABLE. "
                "Base ALL causal statements STRICTLY on quantitative metrics: margin deltas, "
                "cost ratios, FCF conversion rates, leverage ratios. "
                "FORBIDDEN: any product name, segment name, demand narrative, or business unit reference. "
                "ALLOWED: 'Gross margin compressed Xpp', 'Operating leverage deteriorated', "
                "'R&D intensity at X% of revenue', 'FCF conversion at X% of net income'."
            )


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE #2: MDA GROUNDER — Deep Text-Metric Semantic Alignment
# ══════════════════════════════════════════════════════════════════════════════

class MDAGrounder:
    """
    Maps metric changes to specific MD&A narrative evidence.
    Produces cross-modal alignment between numbers and management explanation.
    """

    ALIGNMENT_MAP = [
        # (metric_condition, mda_keywords, grounded_template)
        ("margin_decline",   ["cost", "expense", "higher", "increase", "pressure"],
         "Margin decline consistent with MD&A references to elevated {mda_driver} costs."),
        ("margin_expansion", ["efficiency", "scale", "leverage", "reduction", "savings"],
         "Margin expansion supported by MD&A commentary on {mda_driver} improvements."),
        ("revenue_growth",   ["demand", "volume", "customer", "market", "growth"],
         "Revenue growth corroborated by MD&A references to {mda_driver}."),
        ("revenue_decline",  ["lower", "decline", "weak", "soft", "headwind"],
         "Revenue decline consistent with MD&A references to {mda_driver} headwinds."),
        ("capex_increase",   ["expansion", "invest", "capacity", "infrastructure", "build"],
         "CapEx increase aligned with MD&A investment narrative: {mda_driver}."),
        ("debt_increase",    ["acqui", "borrow", "financing", "credit", "facility"],
         "Leverage increase linked to MD&A financing activity: {mda_driver}."),
    ]

    @classmethod
    def _classify_metric_state(cls, changes: dict, metrics: dict) -> list[str]:
        states = []
        rev_d = changes.get("revenue_yoy_pct", 0) or 0
        gm_d  = changes.get("gross_margin_pct_delta", 0) or 0
        cap   = changes.get("capex", 0) or 0  # not in changes but checked via metrics
        dbt_d = changes.get("debt_yoy_pct", 0) or 0

        if rev_d > 3:   states.append("revenue_growth")
        if rev_d < -3:  states.append("revenue_decline")
        if gm_d < -1:   states.append("margin_decline")
        if gm_d > 1:    states.append("margin_expansion")
        if dbt_d > 10:  states.append("debt_increase")
        return states

    @classmethod
    def build_grounded_context(cls, metrics: dict, changes: dict, mda_text: str) -> str:
        """
        Returns a grounded narrative context block for injection into the prompt.
        Only includes claims with explicit MD&A evidence.
        """
        if not mda_text or len(mda_text) < 150:
            return "[MD&A not available — quantitative causality only]"

        states  = cls._classify_metric_state(changes, metrics)
        mda_low = mda_text.lower()
        grounded_statements = []

        for state in states:
            for condition, keywords, template in cls.ALIGNMENT_MAP:
                if condition != state:
                    continue
                found_kw = [kw for kw in keywords if kw in mda_low]
                if found_kw:
                    driver_str = ", ".join(found_kw[:3])
                    stmt = template.format(mda_driver=driver_str)
                    grounded_statements.append(stmt)
                    break  # one statement per state

        if not grounded_statements:
            return "[MD&A available but no strong metric-narrative alignment detected — use quantitative causality]"

        return "MD&A ALIGNED CONTEXT (use these in your analysis):\n" + "\n".join(
            f"• {s}" for s in grounded_statements
        )


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE #6: EVIDENCE CONFIDENCE SCORER
# ══════════════════════════════════════════════════════════════════════════════

class EvidenceConfidence:
    """Scores evidence quality to produce calibrated, uncertainty-aware outputs."""

    @classmethod
    def score(cls, metrics: dict, changes: dict, text_sections: dict) -> dict:
        mda  = text_sections.get("mda", "") or ""
        risk = text_sections.get("risk_factors", "") or ""
        has_mda  = len(mda) > 200
        has_risk = len(risk) > 200

        # Metric completeness
        key_metrics = ["revenue", "gross_profit", "operating_income", "net_income",
                       "free_cash_flow", "total_assets", "long_term_debt"]
        metric_coverage = sum(1 for m in key_metrics if m in metrics) / len(key_metrics)

        # Change coverage
        key_changes = ["revenue_yoy_pct", "gross_margin_pct_delta", "net_income_yoy_pct"]
        change_coverage = sum(1 for c in key_changes if c in changes) / len(key_changes)

        # Evidence level
        if metric_coverage >= 0.85 and change_coverage >= 0.8 and has_mda:
            level = "HIGH"
            qualifier = ""
        elif metric_coverage >= 0.6 and change_coverage >= 0.5:
            level = "MEDIUM"
            qualifier = "Note: Partial metric coverage — some conclusions may be limited."
        else:
            level = "LOW"
            qualifier = "IMPORTANT: Limited data available. Conclusions are preliminary."

        return {
            "level": level,
            "metric_coverage": round(metric_coverage, 2),
            "change_coverage": round(change_coverage, 2),
            "has_mda": has_mda,
            "has_risk": has_risk,
            "qualifier": qualifier,
        }


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE #12: QUALITY VALIDATION PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

class QualityValidator:
    """
    Automated 4-axis quality validation:
    A) Hallucination detector (unsupported causal claims)
    B) Delta consistency (stated vs computed % changes)
    C) Style validator (filler tokens, verbosity)
    D) Diversity validator (reasoning pattern variety)
    """

    HALLUCINATION_PATTERNS = [
        r"\b(iphone|ipad|mac|services|aws|azure|ads)\b.{0,60}\b(drove|boosted|caused|led to)\b",
        r"\b(strong|robust|surging) demand\b",
        r"\boperational efficiency (drove|boosted|improved) (revenue|margin)\b",
        r"\bmarket (expansion|penetration|share gains) (drove|boosted|caused)\b",
        r"\b(product|customer|geographic) (mix|diversification) (drove|caused|boosted)\b",
    ]
    _hall_compiled = [re.compile(p, re.IGNORECASE) for p in HALLUCINATION_PATTERNS]

    FILLER_PATTERNS = [re.compile(rf"\b{tok}\b", re.IGNORECASE) for tok in FILLER_TOKENS]

    @classmethod
    def check_hallucination(cls, output: str, has_mda: bool) -> dict:
        if has_mda:
            return {"flag": False, "violations": [], "note": "MD&A available — causal claims permitted"}
        violations = [p.search(output).group()[:80] for p in cls._hall_compiled if p.search(output)]
        return {
            "flag": len(violations) > 0,
            "violations": violations,
            "note": f"{len(violations)} unsupported causal claims detected" if violations else "Clean"
        }

    @classmethod
    def check_delta_consistency(cls, output: str, changes: dict) -> dict:
        """Verify that stated numeric % values roughly match computed changes."""
        issues = []
        for field, value in changes.items():
            if not field.endswith("_pct") or value is None:
                continue
            # Look for the field's approximate value in output
            rounded = round(value, 1)
            # Allow ±2pp tolerance for rounding
            pattern = rf"({abs(rounded)-2:.0f}|{abs(rounded)-1:.0f}|{abs(rounded):.0f}|{abs(rounded)+1:.0f}|{abs(rounded)+2:.0f})\s*%"
            if not re.search(pattern, output) and abs(value) > 3:
                issues.append(f"{field}={value:.1f}% not found in output")
        return {"issues": issues, "clean": len(issues) == 0}

    @classmethod
    def check_style(cls, output: str) -> dict:
        filler_hits = [p.pattern for p in cls.FILLER_PATTERNS if p.search(output)]
        word_count  = len(output.split())
        num_count   = len(_NUM_RE.findall(output))
        density     = round(num_count / max(word_count, 1), 3)
        return {
            "filler_tokens": filler_hits[:5],
            "word_count": word_count,
            "numerical_density": density,
            "density_ok": density >= 0.05,   # At least 5% of words are numbers
            "length_ok": 80 <= word_count <= 300,
        }

    @classmethod
    def validate(cls, output: str, metrics: dict, changes: dict,
                 has_mda: bool) -> dict:
        hall = cls.check_hallucination(output, has_mda)
        delta = cls.check_delta_consistency(output, changes)
        style = cls.check_style(output)

        score = 100
        if hall["flag"]:         score -= 20 * len(hall["violations"])
        if not delta["clean"]:   score -= 5 * len(delta["issues"])
        if not style["density_ok"]: score -= 10
        if not style["length_ok"]:  score -= 5
        score = max(0, score)

        return {
            "quality_score": score,
            "hallucination": hall,
            "delta_consistency": delta,
            "style": style,
            "passes_threshold": score >= 70,
        }


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE #8: STRUCTURAL RANDOMIZER
# ══════════════════════════════════════════════════════════════════════════════

class StructuralRandomizer:
    """
    Randomizes output structure ordering to prevent template memorization.
    Preserves logical integrity while breaking surface-level patterns.
    """

    ORDERINGS = [
        # Standard: revenue → profitability → cash → balance sheet
        ["revenue_analysis", "profitability", "profitability_drivers",
         "cash_flow", "balance_sheet", "key_risks", "investment_signal", "signal_rationale"],
        # Cash-first: cash quality leads
        ["cash_flow", "revenue_analysis", "profitability", "profitability_drivers",
         "balance_sheet", "key_risks", "investment_signal", "signal_rationale"],
        # Risk-forward: risks before signal
        ["revenue_analysis", "profitability", "key_risks", "cash_flow",
         "balance_sheet", "profitability_drivers", "investment_signal", "signal_rationale"],
        # Thesis-first: signal up front for conviction
        ["investment_signal", "signal_rationale", "revenue_analysis",
         "profitability", "profitability_drivers", "cash_flow", "balance_sheet", "key_risks"],
    ]

    @classmethod
    def get_ordering(cls, seed: Optional[str] = None) -> list[str]:
        if seed:
            idx = int(hashlib.md5(seed.encode()).hexdigest(), 16) % len(cls.ORDERINGS)
        else:
            idx = random.randint(0, len(cls.ORDERINGS) - 1)
        return cls.ORDERINGS[idx]

    @classmethod
    def build_output_schema_instruction(cls, ordering: list[str]) -> str:
        fields = [f'  "{k}": "<...>"' for k in ordering if k != "key_risks"]
        # key_risks is always an array
        if "key_risks" in ordering:
            idx = ordering.index("key_risks")
            fields.insert(idx, '  "key_risks": ["<risk1>", "<risk2>", "<risk3>"]')
        return "{\n" + ",\n".join(fields) + "\n}"


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE #7: TERM NORMALIZER
# ══════════════════════════════════════════════════════════════════════════════

def normalize_financial_terms(text: str) -> str:
    """Replace non-canonical financial terms with canonical forms."""
    for non_canonical, canonical in CANONICAL_TERMS.items():
        text = re.sub(re.escape(non_canonical), canonical, text, flags=re.IGNORECASE)
    return text


# ══════════════════════════════════════════════════════════════════════════════
# TEXT CLEANING (from v12, enhanced)
# ══════════════════════════════════════════════════════════════════════════════

def clean_section_text(raw: str) -> str:
    lines = raw.replace("\xa0", " ").replace("\u2019", "'").split("\n")
    seen, good = set(), []
    for raw_line in lines:
        line = raw_line.strip()
        if not line or len(line) < MIN_PARA_LEN: continue
        alpha = sum(1 for c in line if c.isalpha())
        if alpha / len(line) < MIN_PARA_ALPHA: continue
        if line.isupper() and len(line) < 120: continue
        if re.search(r"\.\s*\.\s*\.\s*\d+\s*$", line): continue
        key = line[:80].lower()
        if key in seen: continue
        seen.add(key)
        good.append(line)
    return "\n\n".join(good)


def filter_financial_paragraphs(text: str) -> str:
    if not text: return ""
    paras = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    kept  = []
    for para in paras:
        lower    = para.lower()
        kw_hits  = sum(1 for kw in FINANCIAL_KEYWORDS if kw in lower)
        num_hits = len(_NUM_RE.findall(para))
        if kw_hits >= 2 or num_hits >= 2:
            kept.append(para)
    return "\n\n".join(kept)[:MAX_TEXT_CHARS]


def extract_section(text: str, patterns: list[str], doc_len: int) -> str:
    skip_start = max(3000, int(doc_len * 0.05))
    lower      = text.lower()

    def _find(sf: int) -> str:
        start = -1
        for pat in patterns:
            m = re.search(pat, lower[sf:], re.DOTALL)
            if m: start = sf + m.start(); break
        if start < 0: return ""
        tail  = lower[start + 30:]
        end_m = NEXT_ITEM_RE.search(tail)
        end   = (start + 30 + end_m.start()) if end_m else min(start + MAX_TEXT_CHARS * 2, len(text))
        return filter_financial_paragraphs(clean_section_text(text[start:end].strip()))

    r1 = _find(skip_start)
    if r1 and len(r1) >= 200 and len(_NUM_RE.findall(r1)) >= 2:
        return r1[:MAX_TEXT_CHARS]
    r2 = _find(0)
    return (r2 if len(r2) > len(r1) else r1)[:MAX_TEXT_CHARS]


# ─── iXBRL ────────────────────────────────────────────────────────────────────
_RE_IX_HIDDEN = re.compile(r"<ix:hidden[^>]*>.*?</ix:hidden\s*>", re.DOTALL | re.IGNORECASE)
_RE_IX_TAGS   = re.compile(r"</?ix:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>",    re.IGNORECASE)
_RE_XBRL_TAGS = re.compile(r"</?xbrli?:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)
_RE_LINK_TAGS = re.compile(r"</?link:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>",  re.IGNORECASE)

def preprocess_ixbrl(html: str) -> str:
    html = _RE_IX_HIDDEN.sub(" ", html)
    html = _RE_IX_TAGS.sub("",   html)
    html = _RE_XBRL_TAGS.sub("", html)
    html = _RE_LINK_TAGS.sub("", html)
    return html

def html_to_text(html: str) -> str:
    try:
        html  = preprocess_ixbrl(html)
        soup  = BeautifulSoup(html, "lxml")
        for tag in soup(["script","style","head","noscript","meta","link"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        text = re.sub(r"[ \t]{2,}", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()
    except Exception as e:
        logger.warning(f"html_to_text fallback: {e}")
        html = _RE_IX_HIDDEN.sub(" ", html)
        text = re.sub(r"<[^>]+>", " ", html)
        text = re.sub(r"[ \t]{2,}", " ", text)
        return re.sub(r"\n{3,}", "\n\n", text).strip()

def _is_xbrl_json(text: str) -> bool:
    s = text.strip()
    return (s.startswith("{") and
            ('"nsprefix"' in s[:200] or ('"version"' in s[:100] and '"instance"' in s[:200])))


# ─── TICKERS ──────────────────────────────────────────────────────────────────
def get_nasdaq100_tickers() -> list[dict]:
    return [
        {"ticker": "AAPL",  "name": "Apple Inc"},
        {"ticker": "MSFT",  "name": "Microsoft Corp"},
        {"ticker": "NVDA",  "name": "NVIDIA Corp"},
        {"ticker": "GOOGL", "name": "Alphabet Inc"},
        {"ticker": "AMZN",  "name": "Amazon.com Inc"},
        {"ticker": "META",  "name": "Meta Platforms Inc"},
        {"ticker": "TSLA",  "name": "Tesla Inc"},
        {"ticker": "AVGO",  "name": "Broadcom Inc"},
        {"ticker": "COST",  "name": "Costco Wholesale Corp"},
        {"ticker": "NFLX",  "name": "Netflix Inc"},
        {"ticker": "AMD",   "name": "Advanced Micro Devices"},
        {"ticker": "QCOM",  "name": "Qualcomm Inc"},
        {"ticker": "ADBE",  "name": "Adobe Inc"},
        {"ticker": "AMAT",  "name": "Applied Materials Inc"},
        {"ticker": "INTU",  "name": "Intuit Inc"},
        {"ticker": "MU",    "name": "Micron Technology Inc"},
        {"ticker": "LRCX",  "name": "Lam Research Corp"},
        {"ticker": "KLAC",  "name": "KLA Corp"},
        {"ticker": "PANW",  "name": "Palo Alto Networks"},
        {"ticker": "CRWD",  "name": "CrowdStrike Holdings"},
        {"ticker": "CSCO",  "name": "Cisco Systems Inc"},
        {"ticker": "PEP",   "name": "PepsiCo Inc"},
        {"ticker": "TMUS",  "name": "T-Mobile US Inc"},
        {"ticker": "ADP",   "name": "Automatic Data Processing"},
        {"ticker": "SBUX",  "name": "Starbucks Corp"},
        {"ticker": "GILD",  "name": "Gilead Sciences Inc"},
        {"ticker": "BKNG",  "name": "Booking Holdings Inc"},
        {"ticker": "ISRG",  "name": "Intuitive Surgical Inc"},
        {"ticker": "AMGN",  "name": "Amgen Inc"},
        {"ticker": "ADI",   "name": "Analog Devices Inc"},
        {"ticker": "TXN",   "name": "Texas Instruments Inc"},
        {"ticker": "MELI",  "name": "MercadoLibre Inc"},
        {"ticker": "PYPL",  "name": "PayPal Holdings Inc"},
        {"ticker": "ABNB",  "name": "Airbnb Inc"},
        {"ticker": "DASH",  "name": "DoorDash Inc"},
        {"ticker": "FTNT",  "name": "Fortinet Inc"},
        {"ticker": "NET",   "name": "Cloudflare Inc"},
        {"ticker": "MNST",  "name": "Monster Beverage Corp"},
        {"ticker": "ORLY",  "name": "O'Reilly Automotive Inc"},
        {"ticker": "PAYX",  "name": "Paychex Inc"},
        {"ticker": "PCAR",  "name": "PACCAR Inc"},
        {"ticker": "SPOT",  "name": "Spotify Technology SA"},
        {"ticker": "TSCO",  "name": "Tractor Supply Co"},
        {"ticker": "DLTR",  "name": "Dollar Tree Inc"},
        {"ticker": "EA",    "name": "Electronic Arts Inc"},
        {"ticker": "EBAY",  "name": "eBay Inc"},
        {"ticker": "FAST",  "name": "Fastenal Co"},
        {"ticker": "FISV",  "name": "Fiserv Inc"},
        {"ticker": "CDW",   "name": "CDW Corp"},
        {"ticker": "CEG",   "name": "Constellation Energy Corp"},
        {"ticker": "CDNS",  "name": "Cadence Design Systems"},
        {"ticker": "SNPS",  "name": "Synopsys Inc"},
        {"ticker": "MRVL",  "name": "Marvell Technology Inc"},
        {"ticker": "ROST",  "name": "Ross Stores Inc"},
        {"ticker": "ODFL",  "name": "Old Dominion Freight Line"},
        {"ticker": "CTAS",  "name": "Cintas Corp"},
        {"ticker": "VRSK",  "name": "Verisk Analytics Inc"},
        {"ticker": "CPRT",  "name": "Copart Inc"},
        {"ticker": "IDXX",  "name": "IDEXX Laboratories Inc"},
        {"ticker": "CTSH",  "name": "Cognizant Technology Solutions"},
        {"ticker": "DXCM",  "name": "DexCom Inc"},
        {"ticker": "BIIB",  "name": "Biogen Inc"},
        {"ticker": "ILMN",  "name": "Illumina Inc"},
        {"ticker": "REGN",  "name": "Regeneron Pharmaceuticals"},
        {"ticker": "VRTX",  "name": "Vertex Pharmaceuticals"},
        {"ticker": "MRNA",  "name": "Moderna Inc"},
        {"ticker": "ZS",    "name": "Zscaler Inc"},
        {"ticker": "OKTA",  "name": "Okta Inc"},
        {"ticker": "SNOW",  "name": "Snowflake Inc"},
        {"ticker": "DDOG",  "name": "Datadog Inc"},
        {"ticker": "MDB",   "name": "MongoDB Inc"},
        {"ticker": "TTD",   "name": "The Trade Desk Inc"},
        {"ticker": "TEAM",  "name": "Atlassian Corp"},
        {"ticker": "WDAY",  "name": "Workday Inc"},
        {"ticker": "VEEV",  "name": "Veeva Systems Inc"},
        {"ticker": "NOW",   "name": "ServiceNow Inc"},
        {"ticker": "CRM",   "name": "Salesforce Inc"},
        {"ticker": "ORCL",  "name": "Oracle Corp"},
        {"ticker": "IBM",   "name": "IBM Corp"},
        {"ticker": "HPQ",   "name": "HP Inc"},
        {"ticker": "DELL",  "name": "Dell Technologies Inc"},
        {"ticker": "STX",   "name": "Seagate Technology Holdings"},
        {"ticker": "WDC",   "name": "Western Digital Corp"},
        {"ticker": "NTAP",  "name": "NetApp Inc"},
        {"ticker": "KEYS",  "name": "Keysight Technologies Inc"},
        {"ticker": "NXPI",  "name": "NXP Semiconductors NV"},
        {"ticker": "MCHP",  "name": "Microchip Technology Inc"},
        {"ticker": "ON",    "name": "ON Semiconductor Corp"},
        {"ticker": "SWKS",  "name": "Skyworks Solutions Inc"},
        {"ticker": "MPWR",  "name": "Monolithic Power Systems"},
        {"ticker": "ENPH",  "name": "Enphase Energy Inc"},
        {"ticker": "ZM",    "name": "Zoom Video Communications"},
        {"ticker": "DOCU",  "name": "DocuSign Inc"},
        {"ticker": "ALGN",  "name": "Align Technology Inc"},
    ]


# ─── HTTP SESSION ─────────────────────────────────────────────────────────────
class EDGARSession:
    ARCHIVES = "https://www.sec.gov/Archives/edgar/data"
    DATA_BASE = "https://data.sec.gov"
    WWW_BASE  = "https://www.sec.gov"

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update(EDGAR_HEADERS)

    def get(self, url: str, timeout: int = 30, is_html: bool = False) -> Optional[requests.Response]:
        time.sleep(EDGAR_SLEEP)
        hdrs = {**EDGAR_HEADERS, "Accept": "text/html,*/*;q=0.8"} if is_html else EDGAR_HEADERS
        for attempt in range(MAX_RETRIES):
            try:
                r = self.session.get(url, timeout=timeout, headers=hdrs)
                if r.status_code == 429:
                    wait = BACKOFF_BASE ** (attempt + 2)
                    logger.warning(f"EDGAR 429 → sleep {wait:.0f}s"); time.sleep(wait); continue
                if r.status_code in (403, 404):
                    logger.debug(f"HTTP {r.status_code}: {url[-70:]}"); return None
                r.raise_for_status()
                return r
            except requests.exceptions.Timeout:
                time.sleep(BACKOFF_BASE ** (attempt + 1))
            except Exception as e:
                logger.error(f"GET: {e} | {url[-60:]}"); return None
        return None


# ─── CIK LOOKUP ───────────────────────────────────────────────────────────────
_cik_cache: dict[str, str] = {}
_tickers_data: Optional[dict] = None

def ticker_to_cik(ticker: str, sess: EDGARSession) -> Optional[str]:
    global _tickers_data
    if ticker in _cik_cache: return _cik_cache[ticker]
    if _tickers_data is None:
        r = sess.get(f"{sess.WWW_BASE}/files/company_tickers.json")
        _tickers_data = r.json() if r else {}
        logger.info(f"CIK table: {len(_tickers_data):,} entries")
    for entry in _tickers_data.values():
        if entry.get("ticker","").upper() == ticker.upper():
            cik = str(entry["cik_str"]).zfill(10)
            _cik_cache[ticker] = cik
            return cik
    logger.warning(f"No CIK for {ticker}"); return None


# ─── FILING DISCOVERY ─────────────────────────────────────────────────────────
def get_filings(cik: str, form_type: str, sess: EDGARSession, max_results: int = 10) -> list[dict]:
    url = f"{sess.DATA_BASE}/submissions/CIK{cik}.json"
    r   = sess.get(url)
    if not r: return []
    data   = r.json()
    recent = data.get("filings",{}).get("recent",{})
    cutoff = (datetime.now() - timedelta(days=YEARS_BACK*365)).strftime("%Y-%m-%d")

    rows = list(zip(recent.get("form",[]), recent.get("filingDate",[]),
                    recent.get("reportDate",[]), recent.get("accessionNumber",[]),
                    recent.get("primaryDocument",[])))
    for ef in data.get("filings",{}).get("files",[])[:5]:
        fname = ef.get("name","")
        if not fname: continue
        r2 = sess.get(f"{sess.DATA_BASE}/submissions/{fname}")
        if not r2: continue
        try:
            ed = r2.json()
            rows += list(zip(ed.get("form",[]), ed.get("filingDate",[]),
                             ed.get("reportDate",[]), ed.get("accessionNumber",[]),
                             ed.get("primaryDocument",[])))
        except Exception: pass

    base    = form_type.split("/")[0]
    results = []
    for form, dt, period, accno, doc in rows:
        if base not in form: continue
        if dt < cutoff: continue
        results.append({"form_type": form, "filed_date": dt, "period_of_report": period,
                        "accession_no": accno, "primary_document": doc})
        if len(results) >= max_results: break

    results.reverse()  # oldest → newest
    logger.info(f"  {form_type}: {len(results)} filings")
    return results


# ─── DOCUMENT SELECTION ───────────────────────────────────────────────────────
def _get_index_docs(cik: str, accession_no: str, sess: EDGARSession) -> list[dict]:
    clean = accession_no.replace("-","")
    r = sess.get(f"{sess.ARCHIVES}/{int(cik)}/{clean}/{accession_no}-index.json")
    if not r: return []
    try: return r.json().get("documents",[])
    except Exception: return []

def _is_xbrl_stub(fname: str) -> bool:
    return bool(re.match(r"^R\d+\.htm$", fname, re.IGNORECASE))

def _item_score(plain: str) -> int:
    return len(ITEM_HEADING_RE.findall(plain))

def get_narrative_doc(cik: str, accession_no: str, primary_doc: str,
                      sess: EDGARSession) -> tuple[Optional[str], str]:
    clean = accession_no.replace("-","")
    base  = f"{sess.ARCHIVES}/{int(cik)}/{clean}"

    fp_result = None
    if primary_doc and primary_doc.lower().endswith(".htm"):
        url = f"{base}/{primary_doc}"
        r   = sess.get(url, timeout=120, is_html=True)
        if r and len(r.content) > MIN_NARRATIVE_DOC_BYTES:
            plain = html_to_text(r.text)
            score = _item_score(plain)
            logger.debug(f"  Fast-path ({len(r.content)//1024}KB, score={score}): {primary_doc}")
            if score >= 5:
                return url, r.text
            fp_result = (url, r.text, score)

    docs = _get_index_docs(cik, accession_no, sess)
    candidates = [
        doc.get("document","") for doc in docs
        if doc.get("document","").lower().endswith(".htm")
        and not _is_xbrl_stub(doc.get("document",""))
        and not any(p in doc.get("document","").lower() for p in ["cover","signature"])
    ]

    if not candidates:
        if fp_result: return fp_result[0], fp_result[1]
        logger.warning(f"  No candidates for {accession_no}")
        return None, ""

    best_url, best_html, best_score = "", "", -1
    for dfile in candidates[:6]:
        url = f"{base}/{dfile}"
        r   = sess.get(url, timeout=120, is_html=True)
        if not r or len(r.content) < 10_000: continue
        plain = html_to_text(r.text)
        score = _item_score(plain)
        logger.debug(f"    Candidate score={score} ({len(r.content)//1024}KB): {dfile}")
        if score > best_score:
            best_score = score; best_url = url; best_html = r.text

    if best_score >= 3:
        logger.info(f"  Doc (score={best_score}): {best_url.split('/')[-1]}")
        return best_url, best_html

    if fp_result: return fp_result[0], fp_result[1]
    logger.warning(f"  No narrative doc for {accession_no}")
    return None, ""


def get_8k_content(cik: str, accession_no: str, primary_doc: str, sess: EDGARSession) -> str:
    clean = accession_no.replace("-","")
    base  = f"{sess.ARCHIVES}/{int(cik)}/{clean}"
    docs  = _get_index_docs(cik, accession_no, sess)

    for doc in docs:
        dtype = doc.get("type","").upper()
        dfile = doc.get("document","")
        if "99" in dtype and (dfile.lower().endswith(".htm") or dfile.lower().endswith(".txt")):
            r = sess.get(f"{base}/{dfile}", timeout=60, is_html=True)
            if r and len(r.content) > 1_000:
                text = html_to_text(r.text)
                if _is_xbrl_json(text): continue
                if sum(1 for kw in MATERIAL_8K_KEYWORDS if kw in text.lower()) >= 2:
                    logger.info(f"  8-K EX-{dtype}: {dfile}")
                    return text[:4000]

    for doc in docs:
        dfile = doc.get("document","")
        if not dfile.lower().endswith(".htm") or _is_xbrl_stub(dfile): continue
        r = sess.get(f"{base}/{dfile}", timeout=60, is_html=True)
        if r and len(r.content) > 800:
            text = html_to_text(r.text)
            if _is_xbrl_json(text): continue
            if sum(1 for kw in MATERIAL_8K_KEYWORDS if kw in text.lower()) >= 2:
                logger.info(f"  8-K htm: {dfile}")
                return text[:4000]

    r_txt = sess.get(f"{base}/{accession_no}.txt", timeout=60)
    if r_txt:
        best = ""
        for block in re.findall(r"<DOCUMENT>(.*?)</DOCUMENT>", r_txt.text, re.DOTALL|re.IGNORECASE):
            tm      = re.search(r"<TEXT>(.*?)</TEXT>", block, re.DOTALL|re.IGNORECASE)
            content = tm.group(1) if tm else block
            content = html_to_text(content) if "<" in content else re.sub(r"\s{3,}", "\n\n", content).strip()
            if _is_xbrl_json(content): continue
            kw = sum(1 for k in MATERIAL_8K_KEYWORDS if k in content.lower())
            if kw > 2 and len(content) > len(best): best = content
        if best:
            logger.info(f"  8-K SGML: {len(best)} chars")
            return best[:4000]

    logger.warning(f"  8-K no content: {accession_no}")
    return ""


def is_material_8k(text: str) -> bool:
    if not text or len(text) < 200: return False
    return sum(1 for kw in MATERIAL_8K_KEYWORDS if kw in text.lower()) >= 3


def scrape_text_sections(cik: str, accession_no: str, primary_doc: str,
                          form_type: str, sess: EDGARSession) -> dict[str, str]:
    url, html = get_narrative_doc(cik, accession_no, primary_doc, sess)
    if not html:
        logger.warning(f"  No HTML for {accession_no}")
        return {}
    plain = html_to_text(html)
    score = _item_score(plain)
    if score < 3:
        logger.warning(f"  Low item-heading score ({score}) — may be wrong doc")
    section_map = SECTIONS_10K if "10-K" in form_type else SECTIONS_10Q
    result = {name: extract_section(plain, pats, len(plain)) for name, pats in section_map.items()}
    filled = sum(1 for v in result.values() if v)
    logger.info(f"    Sections: {filled}/{len(section_map)} (score={score})")
    return result


# ─── XBRL METRICS ─────────────────────────────────────────────────────────────
FIELD_MAP = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax",
                "NetSales", "RevenueFromContractWithCustomerIncludingAssessedTax",
                "Revenues", "SalesRevenueNet"],
    "cost_of_revenue":           ["CostOfGoodsAndServicesSold","CostOfRevenue","CostOfGoodsSold"],
    "gross_profit":              ["GrossProfit"],
    "operating_expenses":        ["OperatingExpenses"],
    "research_development":      ["ResearchAndDevelopmentExpense"],
    "selling_general_admin":     ["SellingGeneralAndAdministrativeExpense"],
    "operating_income":          ["OperatingIncomeLoss"],
    "interest_expense":          ["InterestExpense","InterestAndDebtExpense"],
    "pretax_income":             ["IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest"],
    "income_tax":                ["IncomeTaxExpenseBenefit"],
    "net_income":                ["NetIncomeLoss"],
    "eps_basic":                 ["EarningsPerShareBasic"],
    "eps_diluted":               ["EarningsPerShareDiluted"],
    "shares_diluted":            ["WeightedAverageNumberOfDilutedSharesOutstanding"],
    "total_assets":              ["Assets"],
    "current_assets":            ["AssetsCurrent"],
    "total_liabilities":         ["Liabilities"],
    "current_liabilities":       ["LiabilitiesCurrent"],
    "total_equity":              ["StockholdersEquity","StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "cash_and_equivalents":      ["CashAndCashEquivalentsAtCarryingValue","CashCashEquivalentsAndShortTermInvestments","CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"],
    "short_term_investments":    ["ShortTermInvestments","MarketableSecuritiesCurrent"],
    "accounts_receivable":       ["AccountsReceivableNetCurrent"],
    "inventory":                 ["InventoryNet"],
    "goodwill":                  ["Goodwill"],
    "long_term_debt":            ["LongTermDebt","LongTermDebtNoncurrent"],
    "retained_earnings":         ["RetainedEarningsAccumulatedDeficit"],
    "operating_cash_flow":       ["NetCashProvidedByUsedInOperatingActivities"],
    "investing_cash_flow":       ["NetCashProvidedByUsedInInvestingActivities"],
    "financing_cash_flow":       ["NetCashProvidedByUsedInFinancingActivities"],
    "capex":                     ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "depreciation_amortization": ["DepreciationDepletionAndAmortization","DepreciationAndAmortization"],
    "stock_based_compensation":  ["ShareBasedCompensation","AllocatedShareBasedCompensationExpense"],
    "dividends_paid":            ["PaymentsOfDividends","PaymentsOfDividendsCommonStock"],
    "share_repurchases":         ["PaymentsForRepurchaseOfCommonStock"],
}
DEI_MAP = {"employees": ["EntityNumberOfEmployees"], "entity_public_float": ["EntityPublicFloat"]}

def load_xbrl_facts(cik: str, sess: EDGARSession) -> dict:
    r = sess.get(f"{sess.DATA_BASE}/api/xbrl/companyfacts/CIK{cik}.json", timeout=30)
    if not r: return {}
    facts = r.json().get("facts",{})
    gaap  = facts.get("us-gaap",{})
    dei   = facts.get("dei",{})
    logger.info(f"  XBRL: {len(gaap)} GAAP + {len(dei)} DEI")
    return {"us-gaap": gaap, "dei": dei}

def _is_consolidated(e: dict) -> bool:
    return e.get("segment") is None

def xbrl_val(ns: dict, concepts: list, period_end: str, base_form: str) -> Optional[float]:
    try:
        td     = datetime.strptime(period_end, "%Y-%m-%d")
        window = {(td + timedelta(days=d)).strftime("%Y-%m-%d") for d in range(-PERIOD_FUZZ_DAYS, PERIOD_FUZZ_DAYS+1)}
    except Exception:
        window = {period_end}
    for concept in concepts:
        for unit in ("USD","USD/shares","shares","pure"):
            matches = [e for e in ns.get(concept,{}).get("units",{}).get(unit,[])
                       if e.get("end") in window and e.get("form","").startswith(base_form)
                       and e.get("val") is not None and _is_consolidated(e)]
            if matches:
                matches.sort(key=lambda e: e.get("filed",""), reverse=True)
                try: return float(matches[0]["val"])
                except (ValueError, TypeError): pass
    return None

def extract_metrics(xbrl: dict, period_end: str, form_type: str) -> dict:
    gaap = xbrl.get("us-gaap",{})
    dei  = xbrl.get("dei",{})
    bf   = form_type.split("/")[0]
    m = {}
    for field, concepts in FIELD_MAP.items():
        m[field] = xbrl_val(gaap, concepts, period_end, bf)
    for field, concepts in DEI_MAP.items():
        m[field] = xbrl_val(dei, concepts, period_end, bf)

    ocf, capex = m.get("operating_cash_flow"), m.get("capex")
    if ocf is not None and capex is not None: m["free_cash_flow"] = ocf - abs(capex)
    oi, da = m.get("operating_income"), m.get("depreciation_amortization")
    if oi is not None and da is not None: m["ebitda"] = oi + abs(da)

    rev,gp  = m.get("revenue"), m.get("gross_profit")
    ni, ta  = m.get("net_income"), m.get("total_assets")
    eq, dbt = m.get("total_equity"), m.get("long_term_debt")
    csh,oi2 = m.get("cash_and_equivalents"), m.get("operating_income")

    if rev and rev > 0:
        if gp  is not None: m["gross_margin_pct"]     = round(gp/rev*100, 2)
        if ni  is not None: m["net_margin_pct"]       = round(ni/rev*100, 2)
        if oi2 is not None: m["operating_margin_pct"] = round(oi2/rev*100, 2)
        # UPGRADE #4: cost structure ratios
        cor = m.get("cost_of_revenue")
        sga = m.get("selling_general_admin")
        rnd = m.get("research_development")
        if cor is not None: m["cogs_ratio_pct"] = round(cor/rev*100, 2)
        if sga is not None: m["sga_ratio_pct"]  = round(sga/rev*100, 2)
        if rnd is not None: m["rd_ratio_pct"]   = round(rnd/rev*100, 2)
    if ta  and ta>0  and ni is not None: m["return_on_assets_pct"] = round(ni/ta*100, 2)
    if eq  and eq>0  and ni is not None: m["return_on_equity_pct"] = round(ni/eq*100, 2)
    if dbt is not None and csh is not None: m["net_debt"] = dbt - csh
    if eq  and eq>0  and dbt is not None:   m["debt_to_equity"] = round(dbt/eq, 3)
    ca,cl = m.get("current_assets"), m.get("current_liabilities")
    if ca and cl and cl>0: m["current_ratio"] = round(ca/cl, 3)

    # FCF conversion ratio
    ni_v  = m.get("net_income")
    fcf_v = m.get("free_cash_flow")
    if ni_v and ni_v > 0 and fcf_v is not None:
        m["fcf_conversion_ratio"] = round(fcf_v / ni_v, 3)

    return {k: (round(v,4) if isinstance(v,float) and abs(v)>0.01 else v)
            for k,v in m.items() if v is not None}


# ─── YoY CHANGES ──────────────────────────────────────────────────────────────
def compute_changes(current: dict, previous: dict) -> dict:
    PAIRS = [
        ("revenue","revenue_yoy_pct"), ("net_income","net_income_yoy_pct"),
        ("gross_profit","gross_profit_yoy_pct"), ("operating_income","operating_income_yoy_pct"),
        ("ebitda","ebitda_yoy_pct"), ("free_cash_flow","fcf_yoy_pct"),
        ("operating_cash_flow","ocf_yoy_pct"), ("long_term_debt","debt_yoy_pct"),
        ("total_assets","assets_yoy_pct"), ("total_equity","equity_yoy_pct"),
        ("research_development","rd_yoy_pct"), ("eps_diluted","eps_yoy_pct"),
        ("capex","capex_yoy_pct"),
    ]
    c = {}
    for metric, label in PAIRS:
        cur, prev = current.get(metric), previous.get(metric)
        if cur is not None and prev and prev != 0:
            c[label] = round(((cur-prev)/abs(prev))*100, 2)
    for margin in ("gross_margin_pct","net_margin_pct","operating_margin_pct",
                   "cogs_ratio_pct","sga_ratio_pct","rd_ratio_pct"):
        cur, prev = current.get(margin), previous.get(margin)
        if cur is not None and prev is not None:
            c[f"{margin}_delta"] = round(cur-prev, 2)
    rev = c.get("revenue_yoy_pct")
    if rev is not None:
        c["trend_revenue"] = ("hypergrowth" if rev>50 else "strong_growth" if rev>20 else
                              "moderate_growth" if rev>5 else "slowing_growth" if rev>0 else
                              "mild_decline" if rev>-10 else "significant_decline")
    gm = c.get("gross_margin_pct_delta")
    if gm is not None:
        c["trend_margin"] = "expanding" if gm>1 else "compressing" if gm<-1 else "stable"
    fcf = c.get("fcf_yoy_pct")
    if fcf is not None:
        c["trend_fcf"] = "strong_generation" if fcf>20 else "improving" if fcf>0 else "deteriorating"
    dbt = c.get("debt_yoy_pct")
    if dbt is not None:
        c["trend_leverage"] = ("rapidly_increasing" if dbt>30 else "moderately_increasing" if dbt>10 else
                               "stable" if -10<=dbt<=10 else "decreasing")
    return c


def compute_signals(m: dict, c: dict) -> dict:
    s = {}
    nm = m.get("net_margin_pct", 0)
    s["profitability_tier"] = ("exceptional" if nm>30 else "strong" if nm>15 else
                               "average" if nm>5 else "weak" if nm>0 else "loss_making")
    ni  = m.get("net_income",    0) or 0
    fcf = m.get("free_cash_flow",0) or 0
    if ni > 0:
        r = fcf/ni
        s["cash_conversion"] = "excellent" if r>1.1 else "good" if r>0.8 else "moderate" if r>0.5 else "poor"
    dr, cr = m.get("debt_to_equity"), m.get("current_ratio")
    if dr is not None:
        s["leverage_health"] = ("minimal_debt" if dr<0.3 else "conservative" if dr<1.0 else
                                "moderate_leverage" if dr<2.0 else "high_leverage" if dr<4.0 else "overleveraged")
    if cr is not None:
        s["liquidity_health"] = ("very_strong" if cr>3.0 else "strong" if cr>2.0 else
                                 "adequate" if cr>1.0 else "tight")
    rev_g = c.get("revenue_yoy_pct",0) or 0
    op_m  = m.get("operating_margin_pct",0) or 0
    r40   = rev_g + op_m
    s["rule_of_40_score"] = round(r40, 1)
    s["rule_of_40_pass"]  = r40 >= 40
    rev = m.get("revenue",0) or 0
    rd  = m.get("research_development",0) or 0
    if rev > 0 and rd > 0:
        rdi = rd/rev*100
        s["rd_intensity_pct"]     = round(rdi, 2)
        s["innovation_investment"] = ("heavy" if rdi>20 else "moderate" if rdi>10 else
                                      "light" if rdi>3 else "minimal")
    # UPGRADE #5: edge case detection
    edge_cases = []
    for name, archetype in EDGE_CASE_ARCHETYPES.items():
        try:
            if archetype["condition"](m, c):
                edge_cases.append(name)
        except Exception:
            pass
    if edge_cases:
        s["edge_cases_detected"] = edge_cases
    return s


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE #10: MULTI-HORIZON REASONING BUILDER
# ══════════════════════════════════════════════════════════════════════════════

def build_multi_horizon_block(company_records: list[dict], form_type: str) -> Optional[dict]:
    """
    Compute QoQ, YoY, and 3yr CAGR from sorted records.
    Returns structured horizon dict for prompt injection.
    """
    recs = sorted(
        [r for r in company_records
         if r["meta"].get("form_type") == form_type and r.get("metrics",{}).get("revenue")],
        key=lambda r: r["meta"].get("period_of_report","")
    )
    if len(recs) < 2:
        return None

    latest   = recs[-1]
    prev_rec = recs[-2] if len(recs) >= 2 else None
    oldest   = recs[0]

    horizons = {}

    # QoQ (quarterly momentum — most useful for 10-Q)
    if prev_rec and "10-Q" in form_type:
        cur_rev  = latest["metrics"].get("revenue")
        prev_rev = prev_rec["metrics"].get("revenue")
        if cur_rev and prev_rev and prev_rev != 0:
            qoq = round(((cur_rev - prev_rev) / abs(prev_rev)) * 100, 2)
            horizons["qoq_revenue_pct"] = qoq
            horizons["qoq_label"] = "accelerating" if qoq > 5 else "stable" if qoq > -5 else "decelerating"

    # YoY (standard annual)
    cur_rev  = latest["metrics"].get("revenue")
    prev_rev = prev_rec["metrics"].get("revenue") if prev_rec else None
    if cur_rev and prev_rev and prev_rev != 0:
        yoy = round(((cur_rev - prev_rev) / abs(prev_rev)) * 100, 2)
        horizons["yoy_revenue_pct"] = yoy

    # 3yr CAGR
    if len(recs) >= 3:
        start_rev = recs[-3]["metrics"].get("revenue")
        end_rev   = latest["metrics"].get("revenue")
        start_per = recs[-3]["meta"].get("period_of_report","")
        end_per   = latest["meta"].get("period_of_report","")
        if start_rev and end_rev and start_rev > 0:
            # Approximate years between
            try:
                y1 = datetime.strptime(start_per[:10], "%Y-%m-%d")
                y2 = datetime.strptime(end_per[:10], "%Y-%m-%d")
                n_years = max((y2 - y1).days / 365.25, 0.5)
                cagr = round(((end_rev / start_rev) ** (1 / n_years) - 1) * 100, 2)
                horizons["three_year_cagr_pct"] = cagr
                horizons["three_year_start_period"] = start_per
                horizons["three_year_end_period"]   = end_per
            except Exception:
                pass

    # Margin trajectory
    cur_gm   = latest["metrics"].get("gross_margin_pct")
    old_gm   = oldest["metrics"].get("gross_margin_pct")
    if cur_gm is not None and old_gm is not None:
        horizons["gross_margin_trajectory_pp"] = round(cur_gm - old_gm, 2)
        horizons["gross_margin_start"] = old_gm
        horizons["gross_margin_end"]   = cur_gm

    # Cyclicality — coefficient of variation of revenue
    revenues = [r["metrics"].get("revenue") for r in recs if r["metrics"].get("revenue")]
    if len(revenues) >= 4:
        mean = sum(revenues) / len(revenues)
        std  = (sum((r - mean)**2 for r in revenues) / len(revenues)) ** 0.5
        cv   = round(std / mean * 100, 1) if mean > 0 else 0
        horizons["revenue_volatility_cv_pct"] = cv
        horizons["cyclicality"] = "high" if cv > 20 else "moderate" if cv > 10 else "low"

    return horizons if horizons else None


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE #9: NEGATIVE EXAMPLE BUILDER
# ══════════════════════════════════════════════════════════════════════════════

def build_negative_examples(metrics: dict, changes: dict, text_sections: dict) -> list[dict]:
    """
    Build negative reasoning examples: what CANNOT be concluded from this data.
    Returns list of {bad_conclusion, correct_conclusion, label} dicts.
    """
    mda_text = text_sections.get("mda", "") or ""
    mda_has_text = len(mda_text) > 200

    negatives = []
    for tmpl in NEGATIVE_REASONING_TEMPLATES:
        try:
            # Safely evaluate trigger condition
            ctx = {**metrics, **changes, "mda_has_text": mda_has_text}
            # Replace metric names to avoid KeyError
            trigger = tmpl["trigger"]
            # Simple eval with context
            result = eval(trigger, {"__builtins__": {}}, ctx)
            if result:
                # Format the template strings with actual values
                fmt_ctx = {k: (v if v is not None else 0) for k, v in ctx.items()}
                try:
                    bad   = tmpl["bad_conclusion"]
                    good  = tmpl["correct_conclusion"].format(**fmt_ctx)
                except (KeyError, ValueError):
                    good  = tmpl["correct_conclusion"]
                negatives.append({
                    "bad_conclusion":     bad,
                    "correct_conclusion": good,
                    "label":              tmpl["label"],
                })
        except Exception:
            continue
    return negatives


# ══════════════════════════════════════════════════════════════════════════════
# LLM PROMPTS — v13 full upgrade suite
# ══════════════════════════════════════════════════════════════════════════════

def _usd(v) -> str:
    if v is None: return "N/A"
    if abs(v)>=1e12: return f"${v/1e12:.2f}T"
    if abs(v)>=1e9:  return f"${v/1e9:.1f}B"
    if abs(v)>=1e6:  return f"${v/1e6:.0f}M"
    return f"${v:,.0f}"

def _pct(v, sfx="%") -> str:
    return f"{v:+.1f}{sfx}" if v is not None else "N/A"

def _pct_plain(v) -> str:
    return f"{v:.1f}%" if v is not None else "N/A"


def build_single_prompt(record: dict, prior_record: Optional[dict] = None,
                        multi_horizon: Optional[dict] = None) -> str:
    """
    v13 master prompt:
    - CausalGuard instruction (upgrade #1)
    - MDAGrounder aligned context (upgrade #2)
    - Quantified-only style rules (upgrade #3, #4)
    - Edge case reasoning focus (upgrade #5)
    - Confidence level declaration (upgrade #6)
    - Randomized output schema ordering (upgrade #8)
    - Multi-horizon block if available (upgrade #10)
    """
    m, c, s   = record.get("metrics",{}), record.get("changes",{}), record.get("signals",{})
    txt, meta = record.get("text_sections",{}), record.get("meta",{})
    mda_text  = (txt.get("mda","") or "")
    risk_text = (txt.get("risk_factors","") or "")[:500]
    has_mda   = len(mda_text) > 200

    # UPGRADE #1: Causality instruction
    causality_instr = CausalGuard.build_causality_instruction(m, c, mda_text)

    # UPGRADE #2: MD&A aligned context
    grounded_ctx = MDAGrounder.build_grounded_context(m, c, mda_text)

    # UPGRADE #6: Evidence confidence
    conf = EvidenceConfidence.score(m, c, txt)
    conf_block = (
        f"EVIDENCE CONFIDENCE: {conf['level']} "
        f"(metric_cov={conf['metric_coverage']:.0%}, "
        f"change_cov={conf['change_coverage']:.0%}, "
        f"MD&A={'YES' if conf['has_mda'] else 'NO'})"
    )
    if conf["qualifier"]:
        conf_block += f"\n{conf['qualifier']}"

    # UPGRADE #5: Edge case instruction
    edge_cases = s.get("edge_cases_detected", [])
    edge_instr = ""
    if edge_cases:
        archetypes = [EDGE_CASE_ARCHETYPES[ec] for ec in edge_cases if ec in EDGE_CASE_ARCHETYPES]
        focuses    = [a["reasoning_focus"] for a in archetypes]
        edge_instr = "\nEDGE CASE ALERT — Special reasoning required:\n" + "\n".join(
            f"• {f}" for f in focuses
        )

    # UPGRADE #8: Randomized output ordering
    ordering     = StructuralRandomizer.get_ordering(seed=record.get("id",""))
    schema_block = StructuralRandomizer.build_output_schema_instruction(ordering)

    # Prior period block
    prior_block = ""
    if prior_record and prior_record.get("metrics"):
        pm = prior_record["metrics"]
        pp = prior_record["meta"].get("period_of_report","prior")
        prior_block = (
            f"\n── PRIOR PERIOD ({pp}) ──\n"
            f"Rev: {_usd(pm.get('revenue'))} | GM: {_pct_plain(pm.get('gross_margin_pct'))} | "
            f"NM: {_pct_plain(pm.get('net_margin_pct'))} | FCF: {_usd(pm.get('free_cash_flow'))} | "
            f"EPS: {pm.get('eps_diluted','N/A')} | D/E: {pm.get('debt_to_equity','N/A')}\n"
        )

    # UPGRADE #10: Multi-horizon block
    horizon_block = ""
    if multi_horizon:
        parts = []
        if "qoq_revenue_pct" in multi_horizon:
            parts.append(f"QoQ Rev: {_pct(multi_horizon['qoq_revenue_pct'])} ({multi_horizon.get('qoq_label','')})")
        if "yoy_revenue_pct" in multi_horizon:
            parts.append(f"YoY Rev: {_pct(multi_horizon['yoy_revenue_pct'])}")
        if "three_year_cagr_pct" in multi_horizon:
            parts.append(f"3yr CAGR: {_pct(multi_horizon['three_year_cagr_pct'])}")
        if "gross_margin_trajectory_pp" in multi_horizon:
            parts.append(f"GM trajectory: {multi_horizon.get('gross_margin_start','?')}% → {multi_horizon.get('gross_margin_end','?')}% ({_pct(multi_horizon['gross_margin_trajectory_pp'],'pp')})")
        if "cyclicality" in multi_horizon:
            parts.append(f"Cyclicality: {multi_horizon['cyclicality']} (CV={multi_horizon.get('revenue_volatility_cv_pct','?')}%)")
        if parts:
            horizon_block = "\n── MULTI-HORIZON ──\n" + " | ".join(parts) + "\n"

    return f"""You are a senior equity research analyst. Produce a precise, evidence-grounded financial assessment.

COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | PERIOD: {meta.get('period_of_report')} | FORM: {meta.get('form_type')}
{conf_block}{prior_block}{horizon_block}
── CURRENT PERIOD ({meta.get('period_of_report')}) ──────────────────────────
Revenue:          {_usd(m.get('revenue'))}  YoY: {_pct(c.get('revenue_yoy_pct'))}
Gross Profit:     {_usd(m.get('gross_profit'))}  Margin: {_pct_plain(m.get('gross_margin_pct'))}  Δ: {_pct(c.get('gross_margin_pct_delta'),'pp')}
COGS ratio:       {_pct_plain(m.get('cogs_ratio_pct'))}  Δ: {_pct(c.get('cogs_ratio_pct_delta'),'pp')}
SG&A ratio:       {_pct_plain(m.get('sga_ratio_pct'))}  Δ: {_pct(c.get('sga_ratio_pct_delta'),'pp')}
Operating Income: {_usd(m.get('operating_income'))}  Margin: {_pct_plain(m.get('operating_margin_pct'))}
EBITDA:           {_usd(m.get('ebitda'))}  YoY: {_pct(c.get('ebitda_yoy_pct'))}
Net Income:       {_usd(m.get('net_income'))}  Margin: {_pct_plain(m.get('net_margin_pct'))}  YoY: {_pct(c.get('net_income_yoy_pct'))}
R&D:              {_usd(m.get('research_development'))}  ({_pct_plain(s.get('rd_intensity_pct'))} of rev)  YoY: {_pct(c.get('rd_yoy_pct'))}
EPS (diluted):    {m.get('eps_diluted','N/A')}  YoY: {_pct(c.get('eps_yoy_pct'))}

FCF:              {_usd(m.get('free_cash_flow'))}  YoY: {_pct(c.get('fcf_yoy_pct'))}  FCF/NI: {m.get('fcf_conversion_ratio','N/A')}
OCF:              {_usd(m.get('operating_cash_flow'))}  YoY: {_pct(c.get('ocf_yoy_pct'))}
CapEx:            {_usd(m.get('capex'))}  YoY: {_pct(c.get('capex_yoy_pct'))}  Buybacks: {_usd(m.get('share_repurchases'))}  SBC: {_usd(m.get('stock_based_compensation'))}

Cash:             {_usd(m.get('cash_and_equivalents'))} | Net Debt: {_usd(m.get('net_debt'))}
LT Debt:          {_usd(m.get('long_term_debt'))}  YoY: {_pct(c.get('debt_yoy_pct'))} | D/E: {m.get('debt_to_equity','N/A')} | CR: {m.get('current_ratio','N/A')}
Total Assets:     {_usd(m.get('total_assets'))} | ROA: {_pct_plain(m.get('return_on_assets_pct'))} | ROE: {_pct_plain(m.get('return_on_equity_pct'))}

Profitability tier: {s.get('profitability_tier','N/A')} | Cash conversion: {s.get('cash_conversion','N/A')}
Leverage: {s.get('leverage_health','N/A')} | Liquidity: {s.get('liquidity_health','N/A')}
Rule-of-40: {s.get('rule_of_40_score','N/A')} ({'PASS' if s.get('rule_of_40_pass') else 'FAIL'})
Revenue trend: {c.get('trend_revenue','N/A')} | Margin trend: {c.get('trend_margin','N/A')} | FCF trend: {c.get('trend_fcf','N/A')}
{edge_instr}
── FILING TEXT ────────────────────────────────────────────────────
MD&A (first 800 chars): {mda_text[:800] or '[not available]'}
Risk Factors (500 chars): {risk_text or '[not available]'}

{grounded_ctx}

── ANALYSIS INSTRUCTIONS ──────────────────────────────────────────
STYLE RULES (strictly enforced):
1. Every directional claim MUST include magnitude + direction + timeframe: "Revenue +{c.get('revenue_yoy_pct',0):.1f}% YoY" not "revenue improved"
2. Banned language: "suggesting", "indicating", "reflecting", "demonstrates", "robust", "solid", "strong/weak" without numbers
3. Use canonical terms: "margin pressure" (not "profitability pressure"), "leverage pressure" (not "debt burden")
4. Target 120-180 words total output. High signal density. No filler.

{causality_instr}

Return ONLY valid JSON in this exact field order:
{schema_block}

investment_signal must be one of: STRONG_BUY | BUY | HOLD | REDUCE | SELL
key_risks: 3 items grounded in data or risk text, each with specific metric reference.
signal_rationale: exactly one sentence with minimum 2 specific metrics."""


def build_negative_example_prompt(record: dict, negatives: list[dict]) -> str:
    """
    UPGRADE #9: Build negative reasoning supervision.
    Teaches the model what CANNOT be concluded from the data.
    """
    m, c  = record.get("metrics",{}), record.get("changes",{})
    meta  = record.get("meta",{})

    neg_block = "\n".join(
        f"  SCENARIO {i+1}: Input data shows {neg['label'].replace('_',' ')}.\n"
        f"    ❌ INVALID: \"{neg['bad_conclusion']}\"\n"
        f"    ✓ CORRECT:  \"{neg['correct_conclusion']}\""
        for i, neg in enumerate(negatives)
    )

    return f"""You are a financial reasoning teacher. For each scenario below, explain WHY the invalid conclusion is logically unsupported by the data, and confirm the correct conclusion.

COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | PERIOD: {meta.get('period_of_report')}
Revenue YoY: {_pct(c.get('revenue_yoy_pct'))} | Gross Margin Δ: {_pct(c.get('gross_margin_pct_delta'),'pp')} | Net Margin: {_pct_plain(m.get('net_margin_pct'))}
FCF YoY: {_pct(c.get('fcf_yoy_pct'))} | D/E: {m.get('debt_to_equity','N/A')} | Current Ratio: {m.get('current_ratio','N/A')}

NEGATIVE REASONING SCENARIOS:
{neg_block}

Return ONLY valid JSON:
{{
  "negative_examples": [
    {{
      "scenario_label": "<label>",
      "why_invalid": "<1-2 sentences: why the bad conclusion is logically unsupported>",
      "correct_reasoning": "<1-2 sentences: what the data actually supports, with specific numbers>",
      "reasoning_boundary": "<1 sentence: what additional evidence WOULD be required to support the invalid claim>"
    }}
  ],
  "epistemic_principle": "<1 sentence: the general reasoning principle demonstrated by these examples>"
}}"""


def build_multiyear_prompt(ticker: str, company: str, form_type: str,
                            timeline: list[dict], multi_horizon: Optional[dict] = None) -> str:
    lines = []
    for entry in timeline:
        p, m, c = entry["period"], entry["metrics"], entry["changes"]
        rev_yoy = f" ({_pct(c.get('revenue_yoy_pct'))} YoY)" if c.get("revenue_yoy_pct") is not None else ""
        lines.append(
            f"{p}: Rev={_usd(m.get('revenue'))}{rev_yoy} | "
            f"GM={_pct_plain(m.get('gross_margin_pct'))} | NM={_pct_plain(m.get('net_margin_pct'))} | "
            f"FCF={_usd(m.get('free_cash_flow'))} | {c.get('trend_revenue','N/A')} | Margin={c.get('trend_margin','N/A')}"
        )

    horizon_block = ""
    if multi_horizon:
        parts = []
        if "three_year_cagr_pct" in multi_horizon:
            parts.append(f"3yr Revenue CAGR: {_pct(multi_horizon['three_year_cagr_pct'])}")
        if "gross_margin_trajectory_pp" in multi_horizon:
            parts.append(f"Gross Margin: {multi_horizon.get('gross_margin_start','?')}% → {multi_horizon.get('gross_margin_end','?')}%")
        if "cyclicality" in multi_horizon:
            parts.append(f"Cyclicality: {multi_horizon['cyclicality']} (CV={multi_horizon.get('revenue_volatility_cv_pct','?')}%)")
        if parts:
            horizon_block = "\nPRE-COMPUTED HORIZONS: " + " | ".join(parts) + "\n"

    return f"""You are a senior equity research analyst specializing in multi-year trend analysis.
Analyze the {len(timeline)}-period financial evolution of {company} ({ticker}) — {form_type} filings.
Period: {timeline[0]['period']} → {timeline[-1]['period']}
{horizon_block}
FINANCIAL TIMELINE (oldest → newest):
{chr(10).join(lines)}

CAUSALITY RULE: Base ALL analysis on the timeline numbers above. Do NOT mention specific products.
Use metric-grounded language: CAGR, margin expansion/compression of X pp, FCF/NI conversion ratio.

STYLE RULES: Every claim needs specific numbers. No filler words. Use canonical terms.
Example of correct style: "Revenue grew from $X to $Y over N periods (CAGR +Z%), while gross margin compressed Xpp."

Return ONLY valid JSON:
{{
  "trend_summary": "<Revenue CAGR with exact start/end values, margin trajectory with exact pp change — 2 sentences max>",
  "key_inflection_points": [
    "<period: specific metric change with exact numbers and evidence-grounded cause>",
    "<period: next inflection with numbers>"
  ],
  "revenue_cagr": "<CAGR% from {timeline[0]['period']} to {timeline[-1]['period']}: show formula (end/start)^(1/n)-1>",
  "margin_evolution": "<gross and net margin: exact start% → end%, Δpp, trend classification>",
  "cash_generation_quality": "<FCF trend with exact values, CapEx as % of rev, FCF/NI conversion>",
  "risk_evolution": "<leverage trend with exact D/E start→end, liquidity changes — numbers only>",
  "investment_thesis": "<bull case: specific metric with value | bear case: specific metric risk with value>",
  "overall_signal": "IMPROVING | STABLE | DETERIORATING | MIXED"
}}"""


def _parse_429_wait(err_str: str) -> int:
    m = re.search(r"try again in (\d+)m([\d.]+)s", err_str)
    if m: return int(m.group(1))*60 + int(float(m.group(2))) + 5
    m = re.search(r"try again in ([\d.]+)s", err_str)
    if m: return int(float(m.group(1))) + 5
    return 60


def generate_analysis(prompt: str, llm) -> str:
    for attempt in range(3):
        try:
            text = llm.invoke(prompt).content.strip()
            m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
            if m: text = m.group(1)
            json.loads(text)
            # UPGRADE #7: normalize canonical terms in output
            text = normalize_financial_terms(text)
            return text
        except json.JSONDecodeError:
            return text if text else ""
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower():
                wait = _parse_429_wait(err)
                if wait > LLM_429_MAX_WAIT:
                    logger.warning(f"  429: wait {wait}s > max — skipping")
                    return ""
                logger.warning(f"  429 → sleep {wait}s (attempt {attempt+1}/3)")
                time.sleep(wait)
                continue
            logger.warning(f"  LLM error: {e}")
            return ""
    return ""


# ─── UPGRADE #11: REASONING TYPE CLASSIFIER ────────────────────────────────────
def classify_reasoning_types(metrics: dict, changes: dict,
                              extra_flags: Optional[dict] = None) -> list[str]:
    types = [k for k, fn in REASONING_TYPES.items() if fn(metrics, changes)]
    if extra_flags:
        for flag, active in extra_flags.items():
            if active and flag in REASONING_TYPES and flag not in types:
                types.append(flag)
    return types


# ─── PAIR BUILDERS ────────────────────────────────────────────────────────────

def build_single_pair(record: dict, analysis: str, prior_record: Optional[dict] = None,
                      multi_horizon: Optional[dict] = None,
                      quality_result: Optional[dict] = None) -> dict:
    m, c, s   = record.get("metrics",{}), record.get("changes",{}), record.get("signals",{})
    txt, meta = record.get("text_sections",{}), record.get("meta",{})
    has_text  = any(v for v in txt.values() if v and len(v) > 100)
    risks     = [kw for kw in RISK_KW if kw in
                 ((txt.get("risk_factors","") or "") + (txt.get("mda","") or "")).lower()][:6]

    # UPGRADE #6: confidence
    conf = EvidenceConfidence.score(m, c, txt)

    # UPGRADE #11: reasoning types
    reasoning_types = classify_reasoning_types(m, c)

    # UPGRADE #5: edge cases
    edge_cases = s.get("edge_cases_detected", [])
    if edge_cases:
        reasoning_types.append("edge_case")

    # Prior context in input
    prior_section = ""
    if prior_record and prior_record.get("metrics"):
        pm = prior_record["metrics"]
        pp = prior_record["meta"].get("period_of_report","prior")
        prior_section = (
            f"\nPRIOR ({pp}): Rev={_usd(pm.get('revenue'))} | "
            f"GM={_pct_plain(pm.get('gross_margin_pct'))} | NM={_pct_plain(pm.get('net_margin_pct'))} | "
            f"FCF={_usd(pm.get('free_cash_flow'))} | EPS={pm.get('eps_diluted','N/A')}"
        )

    # UPGRADE #10: horizon block in input
    horizon_section = ""
    if multi_horizon:
        h_parts = []
        if "qoq_revenue_pct"     in multi_horizon: h_parts.append(f"QoQ Rev: {_pct(multi_horizon['qoq_revenue_pct'])}")
        if "yoy_revenue_pct"     in multi_horizon: h_parts.append(f"YoY Rev: {_pct(multi_horizon['yoy_revenue_pct'])}")
        if "three_year_cagr_pct" in multi_horizon: h_parts.append(f"3yr CAGR: {_pct(multi_horizon['three_year_cagr_pct'])}")
        if "cyclicality"         in multi_horizon: h_parts.append(f"Cyclicality: {multi_horizon['cyclicality']}")
        if h_parts:
            horizon_section = "\nHORIZONS: " + " | ".join(h_parts)

    inp = (
        f"COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | "
        f"FORM: {meta.get('form_type')} | PERIOD: {meta.get('period_of_report')}"
        f"{prior_section}{horizon_section}\n\n"
        f"── INCOME ──\n"
        f"Revenue: {_usd(m.get('revenue'))} (YoY {_pct(c.get('revenue_yoy_pct'))})\n"
        f"Gross Margin: {_pct_plain(m.get('gross_margin_pct'))} Δ{_pct(c.get('gross_margin_pct_delta'),'pp')} | "
        f"COGS: {_pct_plain(m.get('cogs_ratio_pct'))} Δ{_pct(c.get('cogs_ratio_pct_delta'),'pp')}\n"
        f"Net Income: {_usd(m.get('net_income'))} | Margin: {_pct_plain(m.get('net_margin_pct'))} | "
        f"YoY: {_pct(c.get('net_income_yoy_pct'))}\n"
        f"EPS: {m.get('eps_diluted','N/A')} (YoY {_pct(c.get('eps_yoy_pct'))})\n\n"
        f"── CASH FLOW ──\n"
        f"FCF: {_usd(m.get('free_cash_flow'))} (YoY {_pct(c.get('fcf_yoy_pct'))}) | FCF/NI: {m.get('fcf_conversion_ratio','N/A')}\n"
        f"CapEx: {_usd(m.get('capex'))} | Buybacks: {_usd(m.get('share_repurchases'))}\n\n"
        f"── BALANCE SHEET ──\n"
        f"Net Debt: {_usd(m.get('net_debt'))} | D/E: {m.get('debt_to_equity','N/A')} | "
        f"CR: {m.get('current_ratio','N/A')} | ROE: {_pct_plain(m.get('return_on_equity_pct'))}\n\n"
        f"── SIGNALS ──\n"
        f"{s.get('profitability_tier','N/A')} | "
        f"Rule-of-40: {s.get('rule_of_40_score','N/A')} ({'PASS' if s.get('rule_of_40_pass') else 'FAIL'}) | "
        f"Trend: {c.get('trend_revenue','N/A')} | Margin: {c.get('trend_margin','N/A')} | FCF: {c.get('trend_fcf','N/A')}"
    )
    if has_text:
        inp += f"\n\n── MD&A ──\n{(txt.get('mda','') or '')[:400]}"
        inp += f"\n\n── RISKS ──\n{(txt.get('risk_factors','') or '')[:250]}"
    else:
        inp += "\n\n[MD&A not available — quantitative analysis only]"

    if edge_cases:
        inp += f"\n\n── EDGE CASES ──\n{', '.join(edge_cases)}"

    return {
        "instruction": "You are a senior financial analyst. Analyze the SEC filing data and return a structured JSON assessment with evidence-grounded causal reasoning.",
        "input":  inp,
        "output": analysis,
        "record_type": "single_period",
        "metadata": {
            "ticker":               meta.get("ticker"),
            "company":              meta.get("company_name"),
            "period":               meta.get("period_of_report"),
            "form_type":            meta.get("form_type"),
            "output_schema":        "v13_full",
            "analysis_mode":        "full" if has_text else "quantitative_only",
            "evidence_confidence":  conf["level"],
            "profitability":        s.get("profitability_tier"),
            "trend_revenue":        c.get("trend_revenue"),
            "trend_margin":         c.get("trend_margin"),
            "trend_fcf":            c.get("trend_fcf"),
            "edge_cases":           edge_cases,
            "reasoning_types":      reasoning_types,        # UPGRADE #11
            "risk_keywords":        risks,
            "rule_of_40_pass":      s.get("rule_of_40_pass"),
            "metrics_count":        len(m),
            "text_sections_filled": sum(1 for v in txt.values() if v and len(v) > 100),
            "has_mda":              has_text,
            "quality_score":        quality_result.get("quality_score") if quality_result else None,
            "hallucination_flag":   quality_result.get("hallucination",{}).get("flag") if quality_result else None,
        },
    }


def build_negative_pair(record: dict, neg_analysis: str, negatives: list[dict]) -> dict:
    """UPGRADE #9: Negative reasoning pair."""
    m, c, meta = record.get("metrics",{}), record.get("changes",{}), record.get("meta",{})
    inp = (
        f"COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | "
        f"FORM: {meta.get('form_type')} | PERIOD: {meta.get('period_of_report')}\n\n"
        f"Revenue YoY: {_pct(c.get('revenue_yoy_pct'))} | "
        f"Gross Margin Δ: {_pct(c.get('gross_margin_pct_delta'),'pp')} | "
        f"Net Margin: {_pct_plain(m.get('net_margin_pct'))}\n"
        f"FCF YoY: {_pct(c.get('fcf_yoy_pct'))} | "
        f"D/E: {m.get('debt_to_equity','N/A')} | CR: {m.get('current_ratio','N/A')}\n\n"
        "TASK: For each scenario, identify what conclusions CANNOT be drawn from this data.\n"
        + "\n".join(f"Scenario {i+1}: {n['label']}" for i, n in enumerate(negatives))
    )
    return {
        "instruction": "You are a financial reasoning teacher. Identify logically unsupported conclusions and explain the correct epistemic boundaries.",
        "input":  inp,
        "output": neg_analysis,
        "record_type": "negative_reasoning",
        "metadata": {
            "ticker":          meta.get("ticker"),
            "company":         meta.get("company_name"),
            "period":          meta.get("period_of_report"),
            "form_type":       meta.get("form_type"),
            "output_schema":   "v13_negative",
            "reasoning_types": ["negative_inference"],       # UPGRADE #11
            "scenarios":       [n["label"] for n in negatives],
        },
    }


def build_multiyear_pair(ticker: str, company: str, form_type: str,
                          timeline: list[dict], analysis: str,
                          multi_horizon: Optional[dict] = None) -> dict:
    lines = []
    for entry in timeline:
        p, m, c = entry["period"], entry["metrics"], entry["changes"]
        lines.append(
            f"{p}: Rev={_usd(m.get('revenue'))} (YoY {_pct(c.get('revenue_yoy_pct'))}) | "
            f"GM={_pct_plain(m.get('gross_margin_pct'))} | NM={_pct_plain(m.get('net_margin_pct'))} | "
            f"FCF={_usd(m.get('free_cash_flow'))}"
        )

    horizon_line = ""
    if multi_horizon and "three_year_cagr_pct" in multi_horizon:
        horizon_line = f"\nPre-computed CAGR: {_pct(multi_horizon['three_year_cagr_pct'])} | Cyclicality: {multi_horizon.get('cyclicality','N/A')}\n"

    inp = (
        f"Analyze the multi-year financial evolution of {company} ({ticker}) "
        f"({form_type} filings, {timeline[0]['period']} → {timeline[-1]['period']}).\n"
        f"{horizon_line}\n"
        "FINANCIAL TIMELINE (oldest → newest):\n" + "\n".join(lines)
    )
    return {
        "instruction": "You are a senior financial analyst specializing in trend analysis. Analyze this multi-year financial evolution with evidence-grounded causality.",
        "input":  inp,
        "output": analysis,
        "record_type": "multi_year",
        "metadata": {
            "ticker":          ticker,
            "company":         company,
            "form_type":       form_type,
            "output_schema":   "v13_multiyear",
            "period_start":    timeline[0]["period"],
            "period_end":      timeline[-1]["period"],
            "n_periods":       len(timeline),
            "reasoning_types": ["multi_horizon", "trend_analysis", "margin_analysis"],  # UPGRADE #11
        },
    }


# ─── MULTI-YEAR RECORDS ────────────────────────────────────────────────────────
def build_multiyear_records(company_records: list[dict], form_type: str,
                             ticker: str, company_name: str,
                             llm, multi_horizon: Optional[dict] = None) -> list[dict]:
    recs = sorted(
        [r for r in company_records
         if r["meta"].get("form_type") == form_type and len(r.get("metrics",{})) >= 5],
        key=lambda r: r["meta"].get("period_of_report","")
    )
    if len(recs) < 3: return []

    def _tl(subset):
        return [{"period": r["meta"]["period_of_report"],
                 "metrics": r["metrics"], "changes": r["changes"], "signals": r["signals"]}
                for r in subset]

    output = []

    # Full window
    if len(recs) >= 4:
        tl       = _tl(recs)
        analysis = generate_analysis(build_multiyear_prompt(ticker, company_name, form_type, tl, multi_horizon), llm)
        if analysis:
            rec_id = hashlib.md5(f"{ticker}_{form_type}_multiyear_{len(recs)}".encode()).hexdigest()[:12]
            output.append({
                "id": rec_id,
                "meta": {"ticker": ticker, "company_name": company_name, "form_type": form_type,
                         "record_type": "multi_year",
                         "period_start": recs[0]["meta"]["period_of_report"],
                         "period_end":   recs[-1]["meta"]["period_of_report"],
                         "n_periods": len(recs)},
                "metrics":{}, "changes":{}, "signals":{}, "text_sections":{},
                "instruction_pair": build_multiyear_pair(ticker, company_name, form_type, tl, analysis, multi_horizon),
            })
            logger.info(f"    ✓ Multi-year ({len(recs)} periods): {len(analysis)} chars")

    # Rolling 3-year windows
    for i in range(len(recs) - 2):
        subset = recs[i:i+3]
        tl     = _tl(subset)
        sp, ep = subset[0]["meta"]["period_of_report"], subset[-1]["meta"]["period_of_report"]
        analysis = generate_analysis(build_multiyear_prompt(ticker, company_name, form_type, tl), llm)
        if analysis:
            rec_id = hashlib.md5(f"{ticker}_{form_type}_3yr_{sp}_{ep}".encode()).hexdigest()[:12]
            output.append({
                "id": rec_id,
                "meta": {"ticker": ticker, "company_name": company_name, "form_type": form_type,
                         "record_type": "multi_year_3yr", "period_start": sp, "period_end": ep, "n_periods": 3},
                "metrics":{}, "changes":{}, "signals":{}, "text_sections":{},
                "instruction_pair": build_multiyear_pair(ticker, company_name, form_type, tl, analysis),
            })
            logger.info(f"    ✓ 3yr {sp}→{ep}: {len(analysis)} chars")

    return output


# ─── MAIN PIPELINE ────────────────────────────────────────────────────────────
def process_company(company: dict, sess: EDGARSession, llm,
                    forms: tuple = ("10-K","10-Q","8-K"),
                    max_per_form: int = 8,
                    done_accessions: set = None,
                    quality_log=None) -> list[dict]:
    ticker = company["ticker"]
    logger.info(f"══ {ticker} ({company['name']}) ══")
    done_accessions = done_accessions or set()

    cik = ticker_to_cik(ticker, sess)
    if not cik: return []

    xbrl = load_xbrl_facts(cik, sess)
    prev_metrics: dict[str, dict] = {}
    prev_records: dict[str, dict] = {}
    all_records:  list[dict]      = []

    for form_type in forms:
        filings = get_filings(cik, form_type, sess, max_per_form)

        for filing in filings:
            accno  = filing["accession_no"]
            period = filing["period_of_report"]
            prim   = filing["primary_document"]

            if accno in done_accessions:
                logger.debug(f"  Skip (done): {accno}")
                continue

            logger.info(f"  ▶ {form_type} | {period} | {accno}")
            rec_id    = hashlib.md5(f"{ticker}_{form_type}_{period}_{accno}".encode()).hexdigest()[:12]
            base_meta = {"ticker": ticker, "company_name": company["name"], "cik": cik,
                         "form_type": form_type, "filed_date": filing["filed_date"],
                         "period_of_report": period,
                         "fiscal_year": int(period[:4]) if period else None, "accession_no": accno}

            # ── 8-K ─────────────────────────────────────────────────────────
            if "8-K" in form_type:
                event_text = get_8k_content(cik, accno, prim, sess)
                record = {"id": rec_id, "meta": base_meta,
                          "metrics":{}, "changes":{}, "signals":{},
                          "text_sections": {"event_text": event_text}}
                if is_material_8k(event_text):
                    analysis = generate_analysis(build_single_prompt(record), llm)
                    if analysis:
                        has_mda = len(event_text) > 200
                        qr = QualityValidator.validate(analysis, {}, {}, has_mda)
                        record["instruction_pair"] = build_single_pair(record, analysis, quality_result=qr)
                        if quality_log and qr:
                            quality_log.write(json.dumps({"ticker": ticker, "period": period,
                                "accno": accno, "form": form_type, **qr}, ensure_ascii=False) + "\n")
                        logger.info(f"    ✓ 8-K material [Q={qr.get('quality_score',0)}]: {len(analysis)} chars")
                else:
                    logger.debug(f"    8-K not material — skip LLM")
                all_records.append(record)
                continue

            # ── 10-K / 10-Q ─────────────────────────────────────────────────
            metrics       = extract_metrics(xbrl, period, form_type) if xbrl else {}
            text_sections = scrape_text_sections(cik, accno, prim, form_type, sess)
            logger.info(f"    Metrics: {len(metrics)} fields")

            prev_key  = f"{ticker}_{form_type}"
            prior_rec = prev_records.get(prev_key)
            changes   = compute_changes(metrics, prev_metrics[prev_key]) if prev_key in prev_metrics else {}
            if metrics: prev_metrics[prev_key] = metrics

            signals = compute_signals(metrics, changes)
            record  = {"id": rec_id, "meta": base_meta,
                       "metrics": metrics, "changes": changes,
                       "signals": signals, "text_sections": text_sections}

            if len(metrics) >= 5:
                # Multi-horizon (partial — will be fully built after all records collected)
                # Pass what we have so far
                analysis = generate_analysis(
                    build_single_prompt(record, prior_record=prior_rec), llm)

                if analysis:
                    has_mda = any(v for v in text_sections.values() if v and len(v) > 100)
                    # UPGRADE #12: quality validation
                    qr = QualityValidator.validate(analysis, metrics, changes, has_mda)
                    if quality_log:
                        quality_log.write(json.dumps({"ticker": ticker, "period": period,
                            "accno": accno, "form": form_type, **qr}, ensure_ascii=False) + "\n")

                    record["instruction_pair"] = build_single_pair(
                        record, analysis, prior_record=prior_rec, quality_result=qr)
                    mode = 'FULL' if has_mda else 'QUANT'
                    logger.info(f"    ✓ LLM [{mode}|Q={qr['quality_score']}]: {len(analysis)} chars")

                    # UPGRADE #9: negative examples (when triggered)
                    mda_text  = (text_sections.get("mda","") or "")
                    negatives = build_negative_examples(metrics, changes, text_sections)
                    if negatives:
                        neg_analysis = generate_analysis(
                            build_negative_example_prompt(record, negatives), llm)
                        if neg_analysis:
                            neg_rec_id = rec_id + "_neg"
                            neg_record = {
                                "id": neg_rec_id, "meta": base_meta,
                                "metrics": metrics, "changes": changes,
                                "signals": signals, "text_sections": text_sections,
                                "instruction_pair": build_negative_pair(record, neg_analysis, negatives),
                            }
                            all_records.append(neg_record)
                            logger.info(f"    ✓ Negative examples: {len(negatives)} scenarios")
                else:
                    record["_needs_llm"] = True
                    logger.warning(f"    ✗ LLM pending")
            else:
                logger.warning(f"    ✗ LLM skipped: only {len(metrics)} metrics")

            prev_records[prev_key] = record
            all_records.append(record)

    # ── Multi-year + multi-horizon ────────────────────────────────────────────
    for form_type in ("10-K","10-Q"):
        try:
            # UPGRADE #10: compute full multi-horizon before building multi-year prompts
            mh = build_multi_horizon_block(all_records, form_type)

            multiyear = build_multiyear_records(
                all_records, form_type, ticker, company["name"], llm, multi_horizon=mh)
            all_records.extend(multiyear)
            if multiyear:
                logger.success(f"  Multi-year {form_type}: +{len(multiyear)} records")
        except Exception as e:
            logger.warning(f"  Multi-year {form_type} failed: {e}")

    logger.success(f"  {ticker}: {len(all_records)} total records")
    return all_records


def build_dataset(tickers: list[dict], output_file: str = OUTPUT_FILE,
                  forms: tuple = ("10-K","10-Q","8-K"),
                  max_per_form: int = 8, resume: bool = True) -> int:
    llm  = get_llm()
    sess = EDGARSession()

    done_accessions: set[str] = set()
    if resume and os.path.exists(output_file):
        with open(output_file) as f:
            for line in f:
                try:
                    accno = json.loads(line).get("meta",{}).get("accession_no","")
                    if accno: done_accessions.add(accno)
                except Exception: pass
        if done_accessions:
            logger.info(f"Resuming — {len(done_accessions)} accessions done")

    logger.info(f"Processing {len(tickers)} companies | forms: {forms}")
    written = 0; pending = 0

    with open(output_file, "a", encoding="utf-8") as out, \
         open(PENDING_FILE, "a", encoding="utf-8") as pf, \
         open(QUALITY_LOG_FILE, "a", encoding="utf-8") as ql:
        for company in tqdm(tickers, desc="Companies"):
            try:
                records = process_company(
                    company=company, sess=sess, llm=llm,
                    forms=forms, max_per_form=max_per_form,
                    done_accessions=done_accessions, quality_log=ql)
                for rec in records:
                    if rec.pop("_needs_llm", False):
                        pf.write(json.dumps(rec, ensure_ascii=False) + "\n"); pf.flush(); pending += 1
                    else:
                        out.write(json.dumps(rec, ensure_ascii=False) + "\n"); out.flush(); written += 1
                logger.info(f"  ✓ {company['ticker']}: {len(records)} records | total: {written}")
            except Exception as e:
                logger.exception(f"  ✗ {company['ticker']} crashed: {e}"); continue

    # Retry pending
    if pending > 0 and os.path.exists(PENDING_FILE):
        logger.info(f"\n{pending} pending — retrying LLM...")
        retried = 0; llm = get_llm()
        with open(PENDING_FILE) as pf_in, open(output_file, "a") as out, \
             open(QUALITY_LOG_FILE, "a") as ql:
            for line in pf_in:
                try:
                    rec = json.loads(line)
                    analysis = generate_analysis(build_single_prompt(rec), llm)
                    if analysis:
                        m  = rec.get("metrics",{})
                        c  = rec.get("changes",{})
                        ht = any(v for v in rec.get("text_sections",{}).values() if v and len(v)>100)
                        qr = QualityValidator.validate(analysis, m, c, ht)
                        ql.write(json.dumps({"retry": True, **qr}, ensure_ascii=False)+"\n")
                        rec["instruction_pair"] = build_single_pair(rec, analysis, quality_result=qr)
                        retried += 1
                    out.write(json.dumps(rec, ensure_ascii=False) + "\n"); out.flush(); written += 1
                except Exception as e: logger.warning(f"  Retry failed: {e}")
        os.remove(PENDING_FILE)
        logger.success(f"Retry: {retried}/{pending} completed")

    logger.success(f"Done. {written} total records → {output_file}")
    logger.success(f"Quality log → {QUALITY_LOG_FILE}")
    return written


# ─── DATASET STATS REPORTER ────────────────────────────────────────────────────
def report_dataset_stats(output_file: str = OUTPUT_FILE, quality_file: str = QUALITY_LOG_FILE):
    """Post-run analysis of dataset composition and quality."""
    if not os.path.exists(output_file):
        print("No dataset file found.")
        return

    records = []
    with open(output_file) as f:
        for line in f:
            try: records.append(json.loads(line))
            except: pass

    total = len(records)
    print(f"\n{'='*65}")
    print(f"DATASET STATS — v13  ({total} total records)")
    print(f"{'='*65}")

    # Record type breakdown
    types = defaultdict(int)
    for r in records:
        rt = r.get("meta",{}).get("record_type") or r.get("record_type","unknown")
        types[rt] += 1
    print("\nRecord types:")
    for k, v in sorted(types.items(), key=lambda x: -x[1]):
        print(f"  {k:<30} {v:>6} ({v/total*100:.1f}%)")

    # Reasoning type coverage (UPGRADE #11)
    all_rtype_counts = defaultdict(int)
    for r in records:
        pair = r.get("instruction_pair",{})
        for rt in pair.get("metadata",{}).get("reasoning_types",[]):
            all_rtype_counts[rt] += 1
    if all_rtype_counts:
        print("\nReasoning type coverage:")
        for k, v in sorted(all_rtype_counts.items(), key=lambda x: -x[1]):
            print(f"  {k:<30} {v:>6}")

    # Edge case coverage (UPGRADE #5)
    edge_counts = defaultdict(int)
    for r in records:
        for ec in r.get("instruction_pair",{}).get("metadata",{}).get("edge_cases",[]):
            edge_counts[ec] += 1
    if edge_counts:
        print("\nEdge case coverage:")
        for k, v in sorted(edge_counts.items(), key=lambda x: -x[1]):
            print(f"  {k:<35} {v:>5}")

    # Evidence confidence distribution (UPGRADE #6)
    conf_counts = defaultdict(int)
    for r in records:
        ec = r.get("instruction_pair",{}).get("metadata",{}).get("evidence_confidence","N/A")
        conf_counts[ec] += 1
    if conf_counts:
        print("\nEvidence confidence:")
        for k, v in sorted(conf_counts.items()):
            print(f"  {k:<10} {v:>6}")

    # Quality score distribution (UPGRADE #12)
    if os.path.exists(quality_file):
        qscores = []
        hall_flags = 0
        with open(quality_file) as f:
            for line in f:
                try:
                    q = json.loads(line)
                    qs = q.get("quality_score")
                    if qs is not None: qscores.append(qs)
                    if q.get("hallucination",{}).get("flag"): hall_flags += 1
                except: pass
        if qscores:
            avg = sum(qscores)/len(qscores)
            below70 = sum(1 for q in qscores if q < 70)
            print(f"\nQuality validation (n={len(qscores)}):")
            print(f"  Avg quality score:      {avg:.1f}/100")
            print(f"  Below threshold (<70):  {below70} ({below70/len(qscores)*100:.1f}%)")
            print(f"  Hallucination flags:    {hall_flags} ({hall_flags/len(qscores)*100:.1f}%)")

    print(f"\n{'='*65}\n")



In [5]:

# ─── ENTRY POINT ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    tickers = get_nasdaq100_tickers()
    tickers = tickers[:3]   # ← remove [:3] for full run

    print("=" * 65)
    print("SEC Fine-Tuning Dataset Builder  v13  (RESEARCH-GRADE)")
    print("=" * 65)
    print(f"  Companies    : {len(tickers)}")
    print(f"  Forms        : 10-K + 10-Q + 8-K")
    print(f"  Output       : {OUTPUT_FILE}")
    print(f"  Quality log  : {QUALITY_LOG_FILE}")



    count = build_dataset(
        tickers=tickers, output_file=OUTPUT_FILE,
        forms=("10-K","10-Q","8-K"), max_per_form=8, resume=True,
    )

    report_dataset_stats(OUTPUT_FILE, QUALITY_LOG_FILE)
    print(f"\nDone: {count} records → {OUTPUT_FILE}")

/tmp/ipykernel_29732/1389142286.py:12: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  return ChatNVIDIA(
21:31:18 | INFO    | Processing 3 companies | forms: ('10-K', '10-Q', '8-K')


SEC Fine-Tuning Dataset Builder  v13  (RESEARCH-GRADE)
  Companies    : 3
  Forms        : 10-K + 10-Q + 8-K
  Output       : sec_finetune_v13.jsonl
  Quality log  : sec_quality_v13.jsonl


Companies:   0%|          | 0/3 [00:00<?, ?it/s]21:31:18 | INFO    | ══ AAPL (Apple Inc) ══
21:31:19 | INFO    | CIK table: 10,359 entries
21:31:19 | INFO    |   XBRL: 503 GAAP + 2 DEI
21:31:20 | INFO    |   10-K: 5 filings
21:31:20 | INFO    |   ▶ 10-K | 2021-09-25 | 0000320193-21-000105
21:31:21 | INFO    |     Sections: 0/6 (score=30)
21:31:21 | INFO    |     Metrics: 47 fields
21:31:51 | INFO    |     ✓ LLM [QUANT|Q=100]: 1168 chars
21:31:51 | INFO    |   ▶ 10-K | 2022-09-24 | 0000320193-22-000108
21:31:52 | INFO    |     Sections: 0/6 (score=29)
21:31:52 | INFO    |     Metrics: 47 fields
21:32:11 | INFO    |     ✓ LLM [QUANT|Q=75]: 1092 chars
21:32:11 | INFO    |   ▶ 10-K | 2023-09-30 | 0000320193-23-000106
21:32:12 | INFO    |     Sections: 0/6 (score=27)
21:32:12 | INFO    |     Metrics: 47 fields
21:32:29 | INFO    |     ✓ LLM [QUANT|Q=85]: 1176 chars
21:32:29 | INFO    |   ▶ 10-K | 2024-09-28 | 0000320193-24-000123
21:32:30 | INFO    |     Sections: 0/6 (score=27)
21:32:30 | 

KeyboardInterrupt: 